##  Fetcher

In [1]:
# ============================================================
# fixed_usdchf_fetcher.py — USD/CHF Multi-Timeframe Data Fetcher
# ============================================================

# ============================================================

import os
import time
import json
import logging
from datetime import datetime, timedelta

import requests
import pandas as pd


# ─── إعدادات ─────────────────────────────────────────────────
# ★ API key من environment variable (مش من الكود مباشرة!)
TWELVE_DATA_API_KEY = os.environ.get("TWELVE_DATA_API_KEY", "").strip()
if not TWELVE_DATA_API_KEY:
    raise RuntimeError(
        "TWELVE_DATA_API_KEY غير محدد.\n"
        "اضبطه أولاً: export TWELVE_DATA_API_KEY='your_key_here'\n"
        "أو في Colab: os.environ['TWELVE_DATA_API_KEY'] = 'your_key'"
    )

SYMBOL              = "USD/CHF"
SYMBOL_SAFE         = "USDCHF"
OUTPUT_DIR          = "."

# ★ حجم آمن على معظم خطط TwelveData (كان 5000 → 400 Error)
BATCH_SIZE          = 800

# ★ زيادة الحد الأقصى لعدد الطلبات لتعويض حجم الـ batch الأصغر
MAX_REQUESTS_PER_TF = 500


# ─── إعدادات كل Timeframe ───────────────────────────────────
TIMEFRAME_CONFIG = {
    "5min": {
        "api_interval":  "5min",  "candle_minutes": 5,
        "target_candles": 630_720, "years_back": 6,  "delay_sec": 0.4,
        "file": f"{OUTPUT_DIR}/{SYMBOL_SAFE}_5min_raw.csv",
    },
    "15min": {
        "api_interval":  "15min", "candle_minutes": 15,
        "target_candles": 280_320, "years_back": 10, "delay_sec": 0.4,
        "file": f"{OUTPUT_DIR}/{SYMBOL_SAFE}_15min_raw.csv",
    },
    "45min": {
        "api_interval":  "45min", "candle_minutes": 45,
        "target_candles": 116_800, "years_back": 14, "delay_sec": 0.4,
        "file": f"{OUTPUT_DIR}/{SYMBOL_SAFE}_45min_raw.csv",
    },
    "1h": {
        "api_interval":  "1h",    "candle_minutes": 60,
        "target_candles": 105_120, "years_back": 12, "delay_sec": 0.4,
        "file": f"{OUTPUT_DIR}/{SYMBOL_SAFE}_1h_raw.csv",
    },
    "2h": {
        "api_interval":  "2h",    "candle_minutes": 120,
        "target_candles": 61_320, "years_back": 12, "delay_sec": 0.4,
        "file": f"{OUTPUT_DIR}/{SYMBOL_SAFE}_2h_raw.csv",
    },
    "4h": {
        "api_interval":  "4h",    "candle_minutes": 240,
        "target_candles": 32_850, "years_back": 15, "delay_sec": 0.4,
        "file": f"{OUTPUT_DIR}/{SYMBOL_SAFE}_4h_raw.csv",
    },
    "1d": {
        "api_interval":  "1day",  "candle_minutes": 1440,
        "target_candles": 7_560,  "years_back": 20, "delay_sec": 0.4,
        "file": f"{OUTPUT_DIR}/{SYMBOL_SAFE}_1d_raw.csv",
    },
    "1W": {
        "api_interval":  "1week", "candle_minutes": 10080,
        "target_candles": 1_560,  "years_back": 30, "delay_sec": 0.4,
        "file": f"{OUTPUT_DIR}/{SYMBOL_SAFE}_1W_raw.csv",
    },
}

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)


# ─── علامات لسبب التوقف — لتشخيص أوضح في اللوج ─────────────
class StopReason:
    OK              = "reached_target"
    PLAN_LIMIT      = "plan_data_limit"
    NO_MORE_DATA    = "no_older_data_available"
    OLDEST_REACHED  = "reached_years_back"
    HTTP_ERROR      = "http_error"
    NETWORK_ERROR   = "network_error"
    EMPTY_RESPONSE  = "empty_response"


class TwelveDataFetcher:
    BASE_URL = "https://api.twelvedata.com/time_series"

    # رسائل من TwelveData تدل على "لا يوجد بيانات أقدم" (سواء بسبب حدود
    # الخطة أو نهاية سجل البيانات فعلاً). لما نلاقي واحدة منها، بنتوقف
    # بسلاسة بدل ما نعتبرها خطأ ونحاول تاني.
    PLAN_LIMIT_KEYWORDS = [
        "no data is available",
        "range allowed",
        "range exceeded",
        "not available for your plan",
        "not found",
        "data limit exceeded",     # كان 'exceeds' فقط — دلوقتي أكتر تحديداً
        "date range exceeds",       # لتفادي match مع 'rate limit exceeded'
        "maximum range",
    ]

    def __init__(self, api_key: str):
        self.api_key = api_key

    def fetch_batch(
        self,
        interval: str,
        start_date: str = None,
        end_date: str = None,
        outputsize: int = BATCH_SIZE,
        _max_retries: int = 3,
    ) -> tuple:
        """
        Returns tuple: (dataframe, stop_reason).
        stop_reason=None means normal success (may still be empty).
        stop_reason=str means "stop the outer loop" — e.g. plan limit reached.
        """
        params = {
            'symbol':     SYMBOL,
            'interval':   interval,
            'apikey':     self.api_key,
            'format':     'JSON',
            'order':      'ASC',
            'outputsize': outputsize,
        }
        if start_date:
            params['start_date'] = start_date
        if end_date:
            params['end_date'] = end_date

        last_error = None
        for attempt in range(1, _max_retries + 1):
            try:
                r = requests.get(self.BASE_URL, params=params, timeout=30)

                # ★ 400 Bad Request — لا يعاد أبداً، السبب دايماً ثابت
                if r.status_code == 400:
                    msg = self._extract_api_message(r)
                    if self._is_plan_limit(msg):
                        # ليس خطأ حقيقي — وصلنا لحد بيانات الخطة
                        logging.info(
                            f"  ℹ️  [{interval}] TwelveData: {msg} "
                            f"— هنعتبر ده نهاية البيانات المتاحة."
                        )
                        return pd.DataFrame(), StopReason.PLAN_LIMIT
                    else:
                        logging.error(
                            f"  ❌ [{interval}] 400 من TwelveData: {msg}"
                        )
                        return pd.DataFrame(), StopReason.HTTP_ERROR

                # ★ 429 Rate Limit — إعادة مع backoff
                if r.status_code == 429:
                    wait = 15 * attempt   # 15s, 30s, 45s
                    logging.warning(
                        f"  ⏸️  [{interval}] Rate limit (429), الانتظار {wait}s "
                        f"(محاولة {attempt}/{_max_retries})"
                    )
                    time.sleep(wait)
                    continue

                # ★ 5xx Server Errors — إعادة مع backoff (مش خطأنا)
                if 500 <= r.status_code < 600:
                    wait = 5 * attempt
                    logging.warning(
                        f"  ⏸️  [{interval}] Server error {r.status_code}, "
                        f"الانتظار {wait}s (محاولة {attempt}/{_max_retries})"
                    )
                    time.sleep(wait)
                    continue

                # ★ 401/403 — API key غلط أو صلاحيات ناقصة — لا يعاد
                if r.status_code in (401, 403):
                    logging.error(
                        f"  ❌ [{interval}] {r.status_code}: تحقق من API key "
                        f"أو صلاحيات الخطة. النص: {r.text[:200]}"
                    )
                    return pd.DataFrame(), StopReason.HTTP_ERROR

                # ★ أي HTTP error آخر — لوق ومحاولة
                if r.status_code != 200:
                    last_error = f"HTTP {r.status_code}: {r.text[:150]}"
                    logging.warning(
                        f"  ⏸️  [{interval}] {last_error} "
                        f"(محاولة {attempt}/{_max_retries})"
                    )
                    time.sleep(2 * attempt)
                    continue

                # ── نجاح: parse response ──
                data = r.json()
                if 'values' not in data:
                    msg = data.get('message', data.get('status', 'unknown'))
                    if self._is_plan_limit(msg):
                        logging.info(
                            f"  ℹ️  [{interval}] TwelveData: {msg} — توقف بأمان."
                        )
                        return pd.DataFrame(), StopReason.PLAN_LIMIT
                    logging.error(f"  ❌ [{interval}] API error: {msg}")
                    return pd.DataFrame(), StopReason.HTTP_ERROR

                df = pd.DataFrame(data['values'])
                if df.empty:
                    return df, None

                if 'volume' not in df.columns:
                    df['volume'] = 1_000_000

                for col in ['open', 'high', 'low', 'close', 'volume']:
                    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

                df['datetime'] = pd.to_datetime(df['datetime'])
                df.sort_values('datetime', inplace=True)
                df.reset_index(drop=True, inplace=True)
                return df, None

            except (requests.exceptions.Timeout,
                    requests.exceptions.ConnectionError) as e:
                last_error = f"Network: {type(e).__name__}"
                logging.warning(
                    f"  ⏸️  [{interval}] {last_error} "
                    f"(محاولة {attempt}/{_max_retries})"
                )
                time.sleep(3 * attempt)
                continue
            except Exception as e:
                last_error = f"Unexpected: {e}"
                logging.error(f"  ❌ [{interval}] {last_error}")
                return pd.DataFrame(), StopReason.NETWORK_ERROR

        # كل المحاولات فشلت
        logging.error(
            f"  ❌ [{interval}] فشل بعد {_max_retries} محاولات: {last_error}"
        )
        return pd.DataFrame(), StopReason.NETWORK_ERROR

    @staticmethod
    def _extract_api_message(response) -> str:
        """اقرأ رسالة الـ API من الرد (مش من الـ status code)."""
        try:
            body = response.json()
            return str(body.get('message', body.get('status', response.text[:200])))
        except Exception:
            return response.text[:200] if response.text else 'no message'

    @classmethod
    def _is_plan_limit(cls, msg: str) -> bool:
        """هل الرسالة معناها 'وصلنا لحد بيانات الخطة'؟"""
        if not msg:
            return False
        msg_lower = msg.lower()
        return any(kw in msg_lower for kw in cls.PLAN_LIMIT_KEYWORDS)


def fetch_timeframe(
    tf_name: str,
    cfg: dict,
    fetcher: TwelveDataFetcher,
) -> pd.DataFrame:
    interval       = cfg['api_interval']
    target         = cfg['target_candles']
    years_back     = cfg['years_back']
    delay          = cfg['delay_sec']
    candle_minutes = cfg['candle_minutes']

    logging.info(f"\n{'='*55}")
    logging.info(f"جلب GBP/USD {tf_name} | interval={interval} | "
                 f"target={target:,} | batch={BATCH_SIZE}")
    logging.info(f"{'='*55}")

    oldest_wanted = datetime.utcnow() - timedelta(days=365 * years_back)
    end_dt        = datetime.utcnow()

    all_batches   = []
    total_candles = 0
    prev_oldest   = None
    req_count     = 0
    stop_reason   = StopReason.OK

    while req_count < MAX_REQUESTS_PER_TF:
        # حجم كل batch بالدقايق حسب حجم الشمعة
        batch_minutes = BATCH_SIZE * candle_minutes
        start_dt      = end_dt - timedelta(minutes=batch_minutes)
        if start_dt < oldest_wanted:
            start_dt = oldest_wanted

        batch, reason = fetcher.fetch_batch(
            interval=interval,
            start_date=start_dt.strftime("%Y-%m-%d %H:%M:%S"),
            end_date=end_dt.strftime("%Y-%m-%d %H:%M:%S"),
            outputsize=BATCH_SIZE,
        )

        req_count += 1

        # ★ توقف صريح لو الـ API قال "خلاص، مفيش بيانات أقدم"
        if reason == StopReason.PLAN_LIMIT:
            stop_reason = StopReason.PLAN_LIMIT
            logging.info(
                f"  ℹ️  توقف عند حد بيانات الخطة/التوفر بعد {req_count} طلب"
            )
            break

        # ★ خطأ شبكة/HTTP حقيقي — نتوقف بأمان مش نستمر بشكل هبل
        if reason in (StopReason.HTTP_ERROR, StopReason.NETWORK_ERROR):
            stop_reason = reason
            logging.error(
                f"  ❌ توقف بسبب {reason} — راجع اللوجز فوق للتفاصيل"
            )
            break

        if batch.empty:
            logging.warning(f"  ⚠️  Batch {req_count}: فارغ — توقف")
            stop_reason = StopReason.EMPTY_RESPONSE
            break

        all_batches.append(batch)
        total_candles += len(batch)
        oldest_in_batch = batch['datetime'].min()
        newest_in_batch = batch['datetime'].max()

        logging.info(
            f"  Batch {req_count:03d}: {len(batch):,} شمعة | "
            f"إجمالي: {total_candles:,}/{target:,} | "
            f"{oldest_in_batch.date()} → {newest_in_batch.date()}"
        )

        # وصلنا للهدف
        if total_candles >= target:
            logging.info(f"  ✅ وصلنا للهدف {target:,}")
            stop_reason = StopReason.OK
            break

        # وصلنا لحد سنوات المطلوب
        if oldest_in_batch <= oldest_wanted:
            logging.info(f"  ✅ وصلنا للحد التاريخي ({years_back} سنة)")
            stop_reason = StopReason.OLDEST_REACHED
            break

        # التاريخ توقف عن الرجوع — لا يوجد بيانات أقدم
        if prev_oldest is not None and oldest_in_batch >= prev_oldest:
            logging.warning(
                f"  ⚠️  التاريخ توقف — لا توجد بيانات أقدم من {oldest_in_batch}"
            )
            stop_reason = StopReason.NO_MORE_DATA
            break

        prev_oldest = oldest_in_batch
        end_dt      = oldest_in_batch - timedelta(seconds=1)
        time.sleep(delay)

    if not all_batches:
        logging.error(
            f"❌ لا توجد بيانات لـ {tf_name} (السبب: {stop_reason})"
        )
        return pd.DataFrame()

    df = pd.concat(all_batches, ignore_index=True)
    df.drop_duplicates(subset='datetime', keep='first', inplace=True)
    df.sort_values('datetime', inplace=True)
    df.reset_index(drop=True, inplace=True)
    df = df[['datetime', 'open', 'high', 'low', 'close', 'volume']].copy()

    logging.info(
        f"  📊 النهائي: {len(df):,} شمعة | "
        f"{df['datetime'].min().date()} → {df['datetime'].max().date()} | "
        f"سبب التوقف: {stop_reason}"
    )
    return df


def update_existing(
    csv_path: str,
    cfg: dict,
    fetcher: TwelveDataFetcher,
) -> pd.DataFrame:
    df_old = pd.read_csv(csv_path)
    df_old['datetime'] = pd.to_datetime(df_old['datetime'])
    last_dt = df_old['datetime'].max()
    logging.info(f"  آخر شمعة موجودة: {last_dt}")

    batch, reason = fetcher.fetch_batch(
        interval=cfg['api_interval'],
        start_date=(last_dt + timedelta(seconds=1)).strftime("%Y-%m-%d %H:%M:%S"),
        end_date=datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
        outputsize=BATCH_SIZE,
    )

    if reason or batch.empty:
        logging.info("  ✅ لا توجد شموع جديدة أو تعذر الجلب")
        return df_old

    df = pd.concat([df_old, batch], ignore_index=True)
    df.drop_duplicates(subset='datetime', keep='last', inplace=True)
    df.sort_values('datetime', inplace=True)
    df.reset_index(drop=True, inplace=True)
    logging.info(f"  ✅ تحديث: +{len(batch):,} شمعة | إجمالي: {len(df):,}")
    return df


def fetch_all_timeframes(
    mode: str = 'smart',
    timeframes: list = None,
):
    """
    mode:
      - 'fresh'  : جلب كامل (يتجاهل الملفات الموجودة)
      - 'update' : تحديث الملفات الموجودة فقط
      - 'smart'  : لو الملف موجود → update، لو لا → fresh
    """
    fetcher = TwelveDataFetcher(TWELVE_DATA_API_KEY)
    tfs = timeframes if timeframes else list(TIMEFRAME_CONFIG.keys())

    summary = []
    for tf in tfs:
        if tf not in TIMEFRAME_CONFIG:
            logging.warning(f"⚠️  timeframe غير معروف: {tf}")
            continue

        cfg      = TIMEFRAME_CONFIG[tf]
        csv_path = cfg['file']

        try:
            if mode == 'fresh' or (mode == 'smart' and not os.path.exists(csv_path)):
                df = fetch_timeframe(tf, cfg, fetcher)
            else:
                df = update_existing(csv_path, cfg, fetcher)

            if not df.empty:
                df.to_csv(csv_path, index=False)
                logging.info(f"  💾 {csv_path} ({len(df):,} rows)")
                summary.append((tf, len(df), 'OK'))
            else:
                summary.append((tf, 0, 'EMPTY'))
        except Exception as e:
            logging.error(f"  ❌ خطأ في {tf}: {e}")
            summary.append((tf, 0, f'ERROR: {e}'))

        # مسافة بين timeframes لتجنب rate limits
        time.sleep(1)

    logging.info("\n" + "="*55)
    logging.info("📋 ملخص الجلب:")
    logging.info("="*55)
    for tf, n, status in summary:
        logging.info(f"  {tf:6}: {n:>10,} صف | {status}")


if __name__ == '__main__':
    fetch_all_timeframes(mode='smart')


/tmp/ipykernel_3055/3159826285.py:286: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  oldest_wanted = datetime.utcnow() - timedelta(days=365 * years_back)
/tmp/ipykernel_3055/3159826285.py:287: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_dt        = datetime.utcnow()
/tmp/ipykernel_3055/3159826285.py:286: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  oldest_wanted = datetime.utcnow() - timedelta(days=365 * years_back)
/tmp/ipykernel_3055/3159826285.py:287: DeprecationWarning: datetime.datetime.utcnow() is depreca

## — Model Training

 `_MODEL_VERSION = "USDCHF-V1-2024"`

In [8]:
# ═══════════════════════════════════════════════════════════════════
# ★★★ USD/CHF (Swissy) Technical Model — V1 (LATEST)
# ═══════════════════════════════════════════════════════════════════
_MODEL_VERSION = "USDCHF-V1-2024"

import requests
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import TimeSeriesSplit
import joblib
import os
import json
from datetime import datetime, timedelta
import time
import random
import logging
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

# ─────────────────────────────────────────────
# ★ المتغيرات الأساسية
# ─────────────────────────────────────────────
TWELVE_DATA_API_KEY = os.environ.get("TWELVE_DATA_API_KEY", "")
SYMBOL       = "USD/CHF"
SYMBOL_SAFE  = "USDCHF"
CSV_RAW_FILE = f"{SYMBOL_SAFE}_5min_raw.csv"
MODELS_DIR   = "models/usdchf_technical_v8"

ATR_MULT_STOP = 1.3  # USD/CHF (Swissy): أهدأ من Cable (~0.65x تقلب)
ATR_MULT_TAKE = 2.2  # USD/CHF: TP أضيق — تقلب أقل
SPREAD_BPS    = 1.8  # USD/CHF: سبريد أوسع شوية من Cable
PIP_SIZE      = 0.0001

# ══════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════

# pip threshold per timeframe — minimum movement to count as signal
PIP_THRESHOLD = {
    # USD/CHF (Swissy) — أهدأ من Cable بـ ~0.65x
    # GBP ATR 5min ~14 pips vs CHF ~8 pips → thresholds أصغر
    "5min":  3,    # GBP=5 (Swissy moves smaller)
    "15min": 7,    # GBP=11
    "45min": 10,   # GBP=16
    "1h":    13,   # GBP=20
    "2h":    18,   # GBP=28
    "4h":    25,   # GBP=38
    "1d":    40,   # GBP=60
    "1W":    15,   # GBP=22
}

# horizon per timeframe
HORIZON = {
    # ★ FIX 5min: كان 5 (25 دقيقة) → 3 (15 دقيقة) — أوضح وأقل ambiguity
    "5min":  3,
    "15min": 2,
    "45min": 2,
    "1h":    1,
    "2h":    1,
    "4h":    1,
    "1d":    1,
    "1W":    2,
}

# ─── GBP/USD (Cable) Technical Config ───
# التعديلات مبنية على دروس مجربة من SMC/ICT GBP/USD:
#   - gap بين buy/sell أوسع (0.30+) لتجنب "المنطقة الرمادية" والـ overtrading
#   - horizons أطول شوية على الفريمات الصغيرة (Cable moves develop slower)
#   - ADX gate أعلى (Cable شهير بالـ false breakouts)
#   - session_filter=False لأن لندن هي جلسة GBP الأساسية أصلاً
#   - htf_features فيها indicators لندن-مركزة
TIMEFRAME_CONFIG = {
    # ═══ 5min: Cable scalping — تقلب عالي، ATR ~14 pips ═══
    "5min": {
        # ═══ V9 HYBRID: V6 flexibility + V8 outlier protection ═══
        # V6 كان يعمل 62 صفقة PF=3.03 (مع bugs). لما شلنا bugs (V7/V8)
        # جاب 6-42 صفقة PF<1. V9: نعيد V6 config بس مع outlier cap
        # + adaptive threshold معتدل (top 20% مش 8%)
        "interval": "5min", "years": 6,
        "buy_threshold": 0.56, "sell_threshold": 0.42,     # ★ V9: gap 0.16 (V6-style معتدل)
        "fold_buy_threshold": 0.54, "fold_sell_threshold": 0.45,
        "max_trades_per_day": 6,     # ★ V9: V6-level (كان 3 في V8 قليل جداً)
        "resample_rule": "5min", "larger_tf": "1h",
        "htf_features": ["rsi","trend_filter","macd_hist","market_structure","session_momentum","atr_pct","adx"],
        "minutes_per_candle": 5, "min_candles": 500,
        "adx_min_entry": 16,         # ★ V9: V6-style (كان 25 في V8 مانع)
        "session_filter": False,     # ★ V9: مطفي (V6-style)
        "atr_mult_stop": 1.5, "atr_mult_take": 2.5,   # ★ CHF: أضيق (أهدأ)
        "require_htf_alignment": False,   # ★ V9: مطفي (كان في V8 يمنع كل الصفقات)
        "use_adaptive_threshold": True, "adaptive_top_pct": 20,  # ★ V9: 8 → 20 معتدل
        "outlier_cap_multiplier": 3.5,   # ★ V9: يمنع outliers بس مش شديد
    },
    # ═══ 15min ═══
    "15min": {
        # ═══ V9 HYBRID: V6 config (469 صفقة PF=1.84) + V8 fixes ═══
        # V6 كان الأفضل هنا حرفياً (469 صفقة PF=1.84)
        # نرجع V6 config بس مع outlier cap + adaptive معتدل
        "interval": "15min", "years": 4,
        "buy_threshold": 0.56, "sell_threshold": 0.42,     # ★ V9: V6-style (gap 0.16)
        "fold_buy_threshold": 0.54, "fold_sell_threshold": 0.45,
        "max_trades_per_day": 5,
        "resample_rule": "15min", "larger_tf": "4h",
        "htf_features": ["rsi", "trend_filter", "atr_pct", "market_structure", "session_momentum", "ema_alignment"],
        "minutes_per_candle": 15, "min_candles": 350,
        "adx_min_entry": 14,         # ★ V9: V6-style
        "atr_mult_stop": 1.3, "atr_mult_take": 2.3,   # ★ CHF: أضيق
        "require_htf_alignment": False,   # ★ V9: مطفي
        "use_adaptive_threshold": True, "adaptive_top_pct": 25,  # ★ V9: 10 → 25
        "outlier_cap_multiplier": 3.5,
    },
    # ═══ 45min ═══
    "45min": {
        # ═══ V9 HYBRID: V6 (299 صفقة PF=1.77) + V8 outlier protection ═══
        # V6 كان الأفضل (299 صفقة PF=1.77 solid)
        "interval": "45min", "years": 4,
        "buy_threshold": 0.56, "sell_threshold": 0.42,     # ★ V9: V6-style
        "fold_buy_threshold": 0.54, "fold_sell_threshold": 0.45,
        "max_trades_per_day": None,
        "resample_rule": "45min", "larger_tf": "4h",
        "htf_features": ["rsi", "trend_filter", "macd_hist", "ema_alignment", "atr_pct"],
        "minutes_per_candle": 45, "min_candles": 180,
        "adx_min_entry": 12,         # ★ CHF: أقل (أهدأ)
        "atr_mult_stop": 1.3, "atr_mult_take": 2.3,
        "require_htf_alignment": False,   # ★ V9: مطفي
        "use_adaptive_threshold": True, "adaptive_top_pct": 25,  # ★ V9: معتدل
        "outlier_cap_multiplier": 3.5,
    },
    # ═══ 1h ═══
    "1h": {
        "interval": "1h", "years": 5,
        "buy_threshold": 0.56, "sell_threshold": 0.40,
        "fold_buy_threshold": 0.54, "fold_sell_threshold": 0.43,
        "max_trades_per_day": None,
        "resample_rule": "1h", "larger_tf": "4h",
        "htf_features": ["rsi", "trend_filter", "market_structure", "sr_dist", "ema_alignment", "atr_pct"],
        "minutes_per_candle": 60, "min_candles": 200,
        "adx_min_entry": 18,
        "atr_mult_stop": 1.3, "atr_mult_take": 2.3,
    },
    # ═══ 2h ═══
    "2h": {
        "interval": "2h", "years": 5,
        "buy_threshold": 0.56, "sell_threshold": 0.38,     # ★ gap 0.24 (SMC lesson: narrow=overtrading)
        "fold_buy_threshold": 0.54, "fold_sell_threshold": 0.42,
        "max_trades_per_day": None,
        "resample_rule": "2h", "larger_tf": "1d",
        "htf_features": ["rsi", "trend_filter", "atr_pct", "ema_alignment", "market_structure"],
        "minutes_per_candle": 120, "min_candles": 120,
        "adx_min_entry": 18,
        "atr_mult_stop": 1.35, "atr_mult_take": 2.3,
    },
    # ═══ 4h ═══
    "4h": {
        "interval": "4h", "years": 7,
        "buy_threshold": 0.56, "sell_threshold": 0.39,     # ★ gap 0.22
        "fold_buy_threshold": 0.54, "fold_sell_threshold": 0.43,
        "max_trades_per_day": None,
        "resample_rule": "4h", "larger_tf": "1d",
        "htf_features": ["rsi", "trend_filter", "market_structure", "sr_dist", "ema_alignment", "atr_pct"],
        # GBP/USD: أوسع من AUD (Cable trend moves stronger)
        "atr_mult_stop": 1.55, "atr_mult_take": 2.6,
        "minutes_per_candle": 240, "min_candles": 80,
        "adx_min_entry": 20,
    },
    # ═══ 1d ═══
    "1d": {
        # ═══ V9: توازن V6 (16 صفقة) و V8 (20 صفقة) ═══
        # الهدف: 30-40 صفقة مع outlier cap متوسط
        "interval": "1d", "years": 25,
        "buy_threshold": 0.54, "sell_threshold": 0.44,     # ★ V9: gap 0.12
        "fold_buy_threshold": 0.52, "fold_sell_threshold": 0.46,
        "max_trades_per_day": None,
        "resample_rule": "1B", "larger_tf": "1W",
        "htf_features": ["rsi", "trend_filter", "ema_alignment", "market_structure"],
        "atr_mult_stop": 1.5, "atr_mult_take": 2.3,   # ★ CHF: أضيق
        "minutes_per_candle": 1440, "min_candles": 80,
        "adx_min_entry": 12,         # ★ V9: بسيط
        "outlier_cap_multiplier": 2.5,   # ★ V9: يبقى شديد للحماية
    },
    # ═══ 1W ═══
    "1W": {
        # ═══ V8: HTF CONTEXT ONLY — accept insufficient data ═══
        # الحقيقة العلمية: 30 سنة * 52 أسبوع = 1560 أسبوع max
        # حتى مع أقل عتبات، النتايج noisy لأن العينة كلها صغيرة
        # V8: نتركه شغال بس نحذر بوضوح — استخدم للـ HTF context فقط
        # ⚠️ لا تعتمد على 1W كإشارة تداول مستقلة
        "interval": "1week", "years": 30,
        "buy_threshold": 0.53, "sell_threshold": 0.45,     # ★ V8: gap 0.10 (كان 0.06)
        "fold_buy_threshold": 0.51, "fold_sell_threshold": 0.47,
        "max_trades_per_day": None,
        "resample_rule": "W-FRI", "larger_tf": None,
        "htf_features": ["rsi","trend_filter","ema_alignment","market_structure"],
        "minutes_per_candle": 10080, "min_candles": 30,
        "adx_min_entry": 0, "session_filter": False,
        "use_adaptive_threshold": True, "adaptive_top_pct": 25,   # ★ V8: 30 → 25
        "outlier_cap_multiplier": 2.5,   # ★ V8: 3.0 → 2.5
        "min_reliable_trades": 15,        # ★ V8 NEW: إذا trades < 15 → is_reliable=False
        "atr_mult_stop": 2.4, "atr_mult_take": 3.8,
    }
}


# ══════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════
XGB_PARAMS = {
    # GBP/USD (Cable) XGB params:
    # - 5min/15min: regularization قوية (Cable whipsaws كتيرة)
    # - 1h/4h: عمق معتدل — Cable trends قوية بس بتنعكس بسرعة
    # - الفيتشورز المستخدمة أكتر من AUD (London-specific sessions)
    "5min": {
        "max_depth": 4, "learning_rate": 0.018, "gamma": 0.55,
        "min_child_weight": 15, "reg_alpha": 1.2, "reg_lambda": 3.2,
        "subsample": 0.55, "colsample_bytree": 0.55,
        "n_estimators": 600
    },
    "15min": {
        "max_depth": 4, "learning_rate": 0.022, "gamma": 0.42,
        "min_child_weight": 12, "reg_alpha": 0.85, "reg_lambda": 2.6,
        "subsample": 0.60, "colsample_bytree": 0.60,
        "n_estimators": 700
    },
    "45min": {
        "max_depth": 5, "learning_rate": 0.028, "gamma": 0.25,
        "min_child_weight": 8, "reg_alpha": 0.50, "reg_lambda": 1.8,
        "subsample": 0.65, "colsample_bytree": 0.65,
        "n_estimators": 850
    },
    "1h": {
        "max_depth": 5, "learning_rate": 0.022, "gamma": 0.20,
        "min_child_weight": 7, "reg_alpha": 0.35, "reg_lambda": 1.5,
        "subsample": 0.70, "colsample_bytree": 0.70,
        "n_estimators": 950
    },
    "2h": {
        "max_depth": 6, "learning_rate": 0.020, "gamma": 0.15,
        "min_child_weight": 6, "reg_alpha": 0.25, "reg_lambda": 1.3,
        "subsample": 0.72, "colsample_bytree": 0.72,
        "n_estimators": 1000
    },
    "4h": {
        "max_depth": 6, "learning_rate": 0.016, "gamma": 0.12,
        "min_child_weight": 5, "reg_alpha": 0.20, "reg_lambda": 1.2,
        "subsample": 0.75, "colsample_bytree": 0.75,
        "n_estimators": 1100
    },
    "1d": {
        "max_depth": 6, "learning_rate": 0.014, "gamma": 0.10,
        "min_child_weight": 4, "reg_alpha": 0.15, "reg_lambda": 1.0,
        "subsample": 0.78, "colsample_bytree": 0.78,
        "n_estimators": 1200
    },
    "1W": {
        "max_depth": 5,
        "learning_rate": 0.015,
        "gamma": 0.08,
        "min_child_weight": 3,
        "reg_alpha": 0.08,
        "reg_lambda": 0.7,
        "subsample": 0.82,
        "colsample_bytree": 0.80,
        "n_estimators": 900
    },
    "default": {
        "max_depth": 5, "learning_rate": 0.02, "gamma": 0.20,
        "min_child_weight": 8, "reg_alpha": 0.30, "reg_lambda": 1.5,
        "subsample": 0.65, "colsample_bytree": 0.65,
        "n_estimators": 800
    }
}

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("training_usdchf_technical_v9.log", encoding='utf-8'),
        logging.StreamHandler()
    ]
)


# ══════════════════════════════════════════════════════════════
# DATA FETCHER
# ══════════════════════════════════════════════════════════════
class TwelveDataFetcher:
    BASE_URL = "https://api.twelvedata.com/time_series"

    def __init__(self, api_key: str):
        self.api_key = api_key

    # ★ V5 FIX: keywords تدل على "لا يوجد بيانات أقدم" (حد الخطة أو انتهاء البيانات)
    # لما نلاقي واحدة منها، نتوقف بسلاسة بدل ما نعتبرها خطأ ونحاول تاني
    PLAN_LIMIT_KEYWORDS = [
        "no data is available", "range allowed", "range exceeded",
        "not available for your plan", "not found",
        "date range exceeds", "maximum range", "data limit exceeded",
    ]

    def fetch_range(self, symbol=SYMBOL, interval="5min",
                    start_date=None, end_date=None, outputsize=800):
        # ★ V5 FIX: outputsize default 5000 → 800
        # الخطط الأساسية بترفض 5000 → 400 Bad Request
        # 800 يعمل على كل الخطط
        params = {
            'symbol': symbol, 'interval': interval,
            'apikey': self.api_key, 'format': 'JSON',
            'order': 'ASC', 'outputsize': outputsize
        }
        if start_date: params['start_date'] = start_date
        if end_date:   params['end_date']   = end_date

        # ★ V5 FIX: 3 محاولات فقط + إخفاء API key من اللوج (كان بيتسرب في الـ URL)
        # Helper: mask sensitive query params in URL for safe logging
        def _safe_url(url_str):
            import re
            return re.sub(r'apikey=[a-zA-Z0-9]+', 'apikey=***MASKED***', url_str)

        # ★ V5 FIX: 3 محاولات فقط (كان 5) + معالجة ذكية للـ 400
        # 400 = طلب غير صحيح (نفس الطلب مش هيرجع صح لو أعدناه)
        # 429 = rate limit (نستنى ونعيد)
        # 5xx = server error (نعيد مع backoff)
        for attempt in range(3):
            try:
                r = requests.get(self.BASE_URL, params=params, timeout=30)

                # ★ V5 FIX: 400 لا يُعاد — السبب دايماً ثابت (حد خطة/معايير غلط)
                if r.status_code == 400:
                    try:
                        body = r.json()
                        msg = body.get('message', body.get('status', r.text[:150]))
                    except Exception:
                        msg = r.text[:150]
                    msg_lower = msg.lower()
                    if any(kw in msg_lower for kw in self.PLAN_LIMIT_KEYWORDS):
                        logging.info(f"[{interval}] وصلنا لحد بيانات الخطة: {msg}")
                    else:
                        logging.error(f"[{interval}] 400 من TwelveData: {msg}")
                    return pd.DataFrame()  # مافيش إعادة على 400

                # 429 rate limit — إعادة مع wait
                if r.status_code == 429:
                    wait = 15 * (attempt + 1)
                    logging.warning(f"[{interval}] Rate limit (429), انتظار {wait}s")
                    time.sleep(wait)
                    continue

                # 5xx server error — إعادة مع backoff
                if 500 <= r.status_code < 600:
                    logging.warning(f"[{interval}] Server error {r.status_code}, retry")
                    time.sleep(5 * (attempt + 1))
                    continue

                # 401/403 — auth مشكلة، مش هيتصلح بالإعادة
                if r.status_code in (401, 403):
                    logging.error(f"[{interval}] {r.status_code}: API key غلط أو صلاحيات ناقصة")
                    return pd.DataFrame()

                r.raise_for_status()
                data = r.json()

                if 'values' not in data:
                    msg = data.get('message', '')
                    msg_lower = msg.lower() if msg else ''
                    if any(kw in msg_lower for kw in self.PLAN_LIMIT_KEYWORDS) or \
                       "no data" in msg_lower or "not available" in msg_lower:
                        logging.info(f"نهاية التاريخ للفريم {interval}")
                    else:
                        logging.error(f"خطأ API: {msg}")
                    return pd.DataFrame()

                df = pd.DataFrame(data['values'])
                if df.empty: return df
                if 'volume' not in df.columns: df['volume'] = 0
                for col in ['open','high','low','close','volume']:
                    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
                df['datetime'] = pd.to_datetime(df['datetime'])
                df.sort_values('datetime', inplace=True)
                df.reset_index(drop=True, inplace=True)
                return df

            except (requests.exceptions.Timeout,
                    requests.exceptions.ConnectionError) as e:
                logging.warning(f"[{interval}] Network error (محاولة {attempt+1}/3): {type(e).__name__}")
                time.sleep(3 * (attempt + 1))
            except Exception as e:
                logging.error(f"[{interval}] Unexpected error: {e}")
                return pd.DataFrame()
        return pd.DataFrame()


def fetch_full_history(target_candles=100000, interval="5min", max_requests=150):
    # ★ V5 FIX: batch_size 5000 → 800 (يتوافق مع كل خطط TwelveData)
    # + max_requests 25 → 150 لتعويض الحجم الأصغر لكل batch
    fetcher = TwelveDataFetcher(TWELVE_DATA_API_KEY)
    all_data, end_date, batch_size = [], datetime.utcnow(), 800

    mult = 60
    if "min" in interval:   mult = int(interval.replace("min","")) * 60
    elif interval.endswith("h"): mult = int(interval.replace("h","")) * 3600
    elif "d" in interval:   mult = 86400
    elif "week" in interval: mult = 604800

    for req in range(max_requests):
        start_date = end_date - timedelta(seconds=batch_size * mult)
        df_b = fetcher.fetch_range(
            symbol=SYMBOL, interval=interval,
            start_date=start_date.strftime("%Y-%m-%d %H:%M:%S"),
            end_date=end_date.strftime("%Y-%m-%d %H:%M:%S"),
            outputsize=batch_size
        )
        if df_b.empty: break
        all_data.append(df_b)
        logging.info(f"~{len(all_data)*batch_size} شمعة ({interval})")
        end_date = df_b['datetime'].min() - timedelta(seconds=1)
        if len(all_data) * batch_size >= target_candles: break
        time.sleep(0.6)

    if not all_data: return pd.DataFrame()
    df = pd.concat(all_data, ignore_index=True)
    df.drop_duplicates(subset='datetime', keep='first', inplace=True)
    df.sort_values('datetime', inplace=True)
    df.reset_index(drop=True, inplace=True)
    if len(df) > target_candles: df = df.iloc[-target_candles:]
    df.to_csv(f"{SYMBOL_SAFE}_{interval}_raw.csv", index=False)
    logging.info(f"✅ {len(df)} شمعة → {SYMBOL_SAFE}_{interval}_raw.csv")
    return df


def get_or_update_data(symbol_safe, interval="5min", target_candles=100000, force_refresh=False):
    csv_path = f"{symbol_safe}_{interval}_raw.csv"
    if os.path.exists(csv_path) and not force_refresh:
        df_old = pd.read_csv(csv_path)
        df_old['datetime'] = pd.to_datetime(df_old['datetime'])
        last_date = df_old['datetime'].max()
        fetcher = TwelveDataFetcher(TWELVE_DATA_API_KEY)
        df_new = fetcher.fetch_range(
            symbol=SYMBOL, interval=interval,
            start_date=(last_date + timedelta(seconds=1)).strftime("%Y-%m-%d %H:%M:%S"),
            end_date=datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
            outputsize=1000
        )
        if df_new.empty: return df_old
        df = pd.concat([df_old, df_new], ignore_index=True)
        df.drop_duplicates(subset='datetime', keep='last', inplace=True)
        df.sort_values('datetime', inplace=True)
        df.to_csv(csv_path, index=False)
        return df
    max_req = 60 if interval in ["1h","4h","1d"] else 25
    return fetch_full_history(target_candles=target_candles, interval=interval, max_requests=max_req)


# ══════════════════════════════════════════════════════════════
# FEATURE ENGINEER
# ══════════════════════════════════════════════════════════════
class FeatureEngineer:

    def __init__(self):
        self.scaler          = RobustScaler()
        self.feature_columns = None
        self.scaler_fitted   = False
        self.selected_cols   = None    # ★ V2: initialized so transform() check works

    def add_time_features(self, df):
        df = df.copy()
        df['hour']        = df['datetime'].dt.hour
        df['day_of_week'] = df['datetime'].dt.dayofweek
        df['month']       = df['datetime'].dt.month
        df['hour_sin']    = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos']    = np.cos(2 * np.pi * df['hour'] / 24)
        df['dow_sin']     = np.sin(2 * np.pi * df['day_of_week'] / 5)
        df['dow_cos']     = np.cos(2 * np.pi * df['day_of_week'] / 5)
        h = df['hour']
        # ★ USD/CHF (Swissy) sessions — أوروبا (زيورخ) + نيويورك
        # European: 07:00-16:00 UTC (بيانات سويسرا/أوروبا)
        # NY:       13:00-21:00 UTC (Fed/US data — الدولار)
        # Overlap:  13:00-16:00 UTC (Europe-US — أعلى سيولة)
        # Asia:     هادئة جداً على Swissy
        df['is_european'] = ((h >= 7)  & (h < 16)).astype(int)   # ★ CHF home region
        df['is_london']   = ((h >= 7)  & (h < 16)).astype(int)   # (alias — أوروبا)
        df['is_ny']       = ((h >= 13) & (h < 21)).astype(int)
        df['is_overlap']  = ((h >= 13) & (h < 16)).astype(int)   # ★ Europe-US overlap
        df['is_asia']     = ((h >= 0)  & (h < 7)).astype(int)    # هادئة على Swissy
        df['is_sydney']   = ((h >= 22) | (h < 7)).astype(int)    # metadata فقط
        df['is_tokyo']    = ((h >= 0)  & (h < 9)).astype(int)    # metadata فقط
        # ─── فيتشورز خاصة بـ USD/CHF ───
        df['is_swiss_open']    = (h == 7).astype(int)            # ★ افتتاح زيورخ
        df['is_swiss_data_window'] = ((h >= 6) & (h < 8)).astype(int)  # ★ Swiss CPI/KOF (06:30-08:00)
        df['is_snb_window']    = ((h >= 7) & (h < 9)).astype(int)      # ★ SNB قرارات (08:30 CET=07:30 UTC)
        df['is_eu_data_window'] = ((h >= 9) & (h < 10)).astype(int)   # ★ EU/German data
        df['is_us_data_window'] = ((h >= 12) & (h < 14)).astype(int)  # ★ US NFP/CPI/Fed
        df['is_swissy_risky']  = (
            ((h == 7)) | ((h >= 6) & (h < 9)) | ((h >= 12) & (h < 14))
        ).astype(int)   # ★ ساعات الأخبار الخطرة (SNB + US)
        df['is_friday']   = (df['day_of_week'] == 4).astype(int)
        df['is_monday']   = (df['day_of_week'] == 0).astype(int)
        df['is_tuesday']  = (df['day_of_week'] == 1).astype(int)
        df['is_thursday'] = (df['day_of_week'] == 3).astype(int)  # ★ SNB عادة الخميس
        # ★ CHF session weights: Overlap (Europe-US) = الأهم، NY قوي (الدولار)
        df['session_momentum'] = (
            df['is_european']  * 2.5   # ★ CHF home region
            + df['is_ny']      * 2.5   # ★ NY قوي (USD leg)
            + df['is_overlap'] * 3.5   # ★ ذروة السيولة Europe-US
            + df['is_asia']    * 0.5   # ★ منخفض جداً على Swissy
        )
        # ★ SNB risk (Thursdays — عادة اجتماعات البنك السويسري)
        df['is_snb_risk']  = df['is_thursday'].astype(int)
        # placeholders للتوافق مع الكود القديم (GBP/AUD fields)
        df['is_boe_risk']  = 0
        df['is_rba_risk']  = 0
        df['is_china_data'] = 0
        df['is_pre_news']   = (
            (h == 12) | (h == 13) | (h == 6) | (h == 7)
        ).astype(int)  # ★ Swiss/EU data + US times
        return df

    def compute_indicators(self, df):
        df = df.copy()
        c, h, l, o, v = df['close'], df['high'], df['low'], df['open'], df['volume']
        n = len(df)

        p14  = min(14,  max(3, n // 6))
        p20  = min(20,  max(4, n // 5))
        p50  = min(50,  max(5, n // 4))
        p200 = min(200, max(10, n // 2))

        for span in [9, 21, p50, min(100, max(8, n // 3)), p200]:
            df[f'ema_{span}'] = c.ewm(span=span, adjust=False).mean()
        df['ema_9']   = c.ewm(span=9,   adjust=False).mean()
        df['ema_21']  = c.ewm(span=21,  adjust=False).mean()
        df['ema_50']  = c.ewm(span=p50, adjust=False).mean()
        df['ema_100'] = c.ewm(span=min(100, max(8, n // 3)), adjust=False).mean()
        df['ema_200'] = c.ewm(span=p200, adjust=False).mean()
        df['sma_20']  = c.rolling(p20).mean()
        df['sma_50']  = c.rolling(p50).mean()

        delta = c.diff()
        gain  = delta.clip(lower=0).rolling(p14).mean()
        loss  = (-delta.clip(upper=0)).rolling(p14).mean()
        df['rsi'] = 100 - (100 / (1 + gain / (loss + 1e-9)))

        rsi_min = df['rsi'].rolling(p14).min()
        rsi_max = df['rsi'].rolling(p14).max()
        df['stoch_rsi'] = (df['rsi'] - rsi_min) / (rsi_max - rsi_min + 1e-9)

        ema12 = c.ewm(span=min(12, max(3, n // 8)), adjust=False).mean()
        ema26 = c.ewm(span=min(26, max(5, n // 5)), adjust=False).mean()
        macd  = ema12 - ema26
        sig   = macd.ewm(span=min(9, max(3, n // 10)), adjust=False).mean()
        df['macd']        = macd
        df['macd_signal'] = sig
        df['macd_hist']   = macd - sig

        tr = pd.concat([h - l,
                        (h - c.shift(1)).abs(),
                        (l - c.shift(1)).abs()], axis=1).max(axis=1)
        df['atr']      = tr.rolling(p14).mean()
        df['atr_pct']  = df['atr'] / (c + 1e-9)
        df['atr_pips'] = df['atr'] / PIP_SIZE

        bb_mid = c.rolling(p20).mean()
        bb_std = c.rolling(p20).std()
        df['bb_upper'] = bb_mid + 2 * bb_std
        df['bb_lower'] = bb_mid - 2 * bb_std
        df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / (bb_mid + 1e-9)
        df['bb_pct']   = (c - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'] + 1e-9)

        tp = (h + l + c) / 3
        df['cci'] = (tp - tp.rolling(p20).mean()) / (0.015 * tp.rolling(p20).std() + 1e-9)

        hh14 = h.rolling(p14).max()
        ll14 = l.rolling(p14).min()
        df['williams_r'] = -100 * (hh14 - c) / (hh14 - ll14 + 1e-9)

        pos_mf = (tp * v).where(tp > tp.shift(1), 0).rolling(p14).sum()
        neg_mf = (tp * v).where(tp < tp.shift(1), 0).rolling(p14).sum()
        df['mfi'] = 100 - (100 / (1 + pos_mf / (neg_mf + 1e-9)))

        plus_dm  = (h - h.shift(1)).clip(lower=0)
        minus_dm = (l.shift(1) - l).clip(lower=0)
        plus_dm  = plus_dm.where(plus_dm  > minus_dm, 0)
        minus_dm = minus_dm.where(minus_dm > plus_dm,  0)
        atr14    = tr.rolling(p14).mean()
        pdi      = 100 * plus_dm.rolling(p14).mean()  / (atr14 + 1e-9)
        mdi      = 100 * minus_dm.rolling(p14).mean() / (atr14 + 1e-9)
        dx       = (pdi - mdi).abs() / (pdi + mdi + 1e-9) * 100
        df['adx']       = dx.rolling(p14).mean()
        df['adx_plus']  = pdi
        df['adx_minus'] = mdi

        s5  = min(5,  max(1, n // 15))
        s10 = min(10, max(2, n // 10))
        s20 = min(20, max(3, n // 7))
        df['momentum_5']  = c - c.shift(s5)
        df['momentum_10'] = c - c.shift(s10)
        df['roc_5']       = c.pct_change(s5)
        df['roc_10']      = c.pct_change(s10)
        df['roc_20']      = c.pct_change(s20)

        body   = (c - o).abs()
        range_ = (h - l).clip(lower=1e-9)
        df['body_ratio'] = body / range_
        df['upper_wick'] = (h - c.clip(lower=o))  / range_
        df['lower_wick'] = (c.clip(upper=o) - l)  / range_
        df['candle_dir'] = np.sign(c - o)

        df['vol_regime']       = c.pct_change().rolling(p20).std()
        df['vol_ratio']        = df['atr'] / (df['atr'].rolling(p50).mean() + 1e-9)
        df['daily_range_pips'] = (h - l) / PIP_SIZE
        df['pip_volatility']   = df['vol_regime'] / PIP_SIZE

        # ★ FIX-5: إضافة ميزات momentum بالـ pips (أوضح للموديل)
        df['momentum_5_pips']  = df['momentum_5']  / PIP_SIZE
        df['momentum_10_pips'] = df['momentum_10'] / PIP_SIZE

        return df

    def add_advanced_volume(self, df):
        df = df.copy()
        c, v, h, l = df['close'], df['volume'], df['high'], df['low']

        df['volume_delta']   = v * np.sign(c.diff())
        df['cvd']            = df['volume_delta'].cumsum()
        df['cvd_slope']      = df['cvd'].diff(5) / (df['cvd'].abs() + 1e-9)
        df['cvd_divergence'] = np.where(df['cvd_slope'] * c.pct_change() < 0, 1, 0)

        df['vpt']       = (v * c.pct_change()).cumsum()
        df['vpt_slope'] = df['vpt'].diff(5) / (df['vpt'].abs() + 1e-9)

        vol_ma20 = v.rolling(20).mean()
        vol_ma50 = v.rolling(50).mean()
        df['vol_ratio_20'] = v / (vol_ma20 + 1e-9)
        df['vol_ratio_50'] = v / (vol_ma50 + 1e-9)
        df['volume_surge'] = (v > vol_ma20 * 2).astype(int)

        bull_vol = v.where(c > c.shift(1), 0)
        bear_vol = v.where(c < c.shift(1), 0)
        df['vol_zone_bull'] = bull_vol.rolling(20).mean() / (vol_ma20 + 1e-9)
        df['vol_zone_bear'] = bear_vol.rolling(20).mean() / (vol_ma20 + 1e-9)
        df['vol_imbalance'] = (
            (bull_vol.rolling(10).sum() - bear_vol.rolling(10).sum())
            / (v.rolling(10).sum() + 1e-9)
        )

        price_up = (c.diff() > 0).astype(int)
        vol_up   = (v.diff() > 0).astype(int)
        df['volume_divergence'] = np.where(price_up != vol_up, 1, 0)

        df['eom']     = (
            ((h + l) / 2 - (h.shift(1) + l.shift(1)) / 2)
            / (v / (h - l + 1e-9) + 1e-9)
        )
        df['eom_sma'] = df['eom'].rolling(14).mean()
        return df

    def add_price_action(self, df):
        df = df.copy()
        o, h, l, c = df['open'], df['high'], df['low'], df['close']
        body       = (c - o).abs()
        range_     = (h - l + 1e-9)
        upper_wick = h - c.clip(lower=o)
        lower_wick = c.clip(upper=o) - l

        df['pinbar_bull'] = ((lower_wick / range_ > 0.6) & (body / range_ < 0.3)).astype(int)
        df['pinbar_bear'] = ((upper_wick / range_ > 0.6) & (body / range_ < 0.3)).astype(int)
        df['pinbar']      = (df['pinbar_bull'] | df['pinbar_bear']).astype(int)

        df['engulfing_bull'] = (
            (c.shift(1) < o.shift(1)) & (c > o)
            & (o <= c.shift(1)) & (c >= o.shift(1))
        ).astype(int)
        df['engulfing_bear'] = (
            (c.shift(1) > o.shift(1)) & (c < o)
            & (o >= c.shift(1)) & (c <= o.shift(1))
        ).astype(int)
        df['engulfing'] = (df['engulfing_bull'] | df['engulfing_bear']).astype(int)

        df['inside_bar']  = ((h < h.shift(1)) & (l > l.shift(1))).astype(int)
        df['outside_bar'] = ((h > h.shift(1)) & (l < l.shift(1))).astype(int)
        df['doji']        = (body / range_ < 0.1).astype(int)

        df['three_white_soldiers'] = (
            (c > o) & (c.shift(1) > o.shift(1)) & (c.shift(2) > o.shift(2))
            & (c > c.shift(1)) & (c.shift(1) > c.shift(2))
        ).astype(int)
        df['three_black_crows'] = (
            (c < o) & (c.shift(1) < o.shift(1)) & (c.shift(2) < o.shift(2))
            & (c < c.shift(1)) & (c.shift(1) < c.shift(2))
        ).astype(int)

        df['gap_up']      = (o > h.shift(1)).astype(int)
        df['gap_down']    = (o < l.shift(1)).astype(int)
        df['price_accel'] = c.diff().diff()
        return df

    def add_trend_filter(self, df):
        df = df.copy()
        c = df['close']
        df['trend_filter']     = (df['ema_50']  > df['ema_200']).astype(int)
        df['trend_filter_9']   = (df['ema_9']   > df['ema_21']).astype(int)
        df['ema_slope_50']     = df['ema_50'].diff(3)  / (c + 1e-9)
        df['ema_slope_200']    = df['ema_200'].diff(3) / (c + 1e-9)
        df['price_vs_ema50']   = (c > df['ema_50']).astype(int)
        df['price_vs_ema200']  = (c > df['ema_200']).astype(int)
        df['price_vs_ema9']    = (c > df['ema_9']).astype(int)
        df['trend_strength']   = (df['ema_50']  - df['ema_200']) / (c + 1e-9)
        df['trend_strength_9'] = (df['ema_9']   - df['ema_21'])  / (c + 1e-9)
        df['ema_alignment']    = (
            (df['ema_9']   > df['ema_21']).astype(int)
            + (df['ema_21']  > df['ema_50']).astype(int)
            + (df['ema_50']  > df['ema_100']).astype(int)
            + (df['ema_100'] > df['ema_200']).astype(int)
        ) / 4.0
        return df

    def add_market_structure(self, df, window=10):
        df = df.copy()
        high, low, c = df['high'], df['low'], df['close']
        rh = high.rolling(window).max()
        rl = low.rolling(window).min()
        prh = rh.shift(window)
        prl = rl.shift(window)

        df['hh'] = (rh > prh).astype(int)
        df['hl'] = (rl > prl).astype(int)
        df['lh'] = (rh < prh).astype(int)
        df['ll'] = (rl < prl).astype(int)
        df['market_structure'] = np.where(
            (df['hh'] == 1) & (df['hl'] == 1), 1,
            np.where((df['lh'] == 1) & (df['ll'] == 1), -1, 0)
        )
        df['bos_up']    = (c > rh.shift(1)).astype(int)
        df['bos_down']  = (c < rl.shift(1)).astype(int)
        df['swing_high'] = (
            (high > high.shift(1)) & (high > high.shift(-1))
            & (high > high.shift(2)) & (high > high.shift(-2))
        ).astype(int)
        df['swing_low'] = (
            (low < low.shift(1)) & (low < low.shift(-1))
            & (low < low.shift(2)) & (low < low.shift(-2))
        ).astype(int)
        return df

    def add_support_resistance(self, df, window=20):
        df = df.copy()
        c  = df['close']
        w   = min(window, max(5, len(df) // 6))
        w50 = min(50, max(5, len(df) // 4))

        df['resistance']         = df['high'].rolling(w).max()
        df['support']            = df['low'].rolling(w).min()
        df['dist_to_resistance'] = (df['resistance'] - c) / (c + 1e-9)
        df['dist_to_support']    = (c - df['support'])    / (c + 1e-9)
        # ★ GBP/USD: نفس pip size (0.0001)
        df['near_resistance']    = (df['dist_to_resistance'] < 0.0005).astype(int)
        df['near_support']       = (df['dist_to_support']    < 0.0005).astype(int)

        sr_range      = df['resistance'] - df['support']
        df['sr_dist'] = (c - df['support']) / (sr_range + 1e-9)
        df['resistance_50'] = df['high'].rolling(w50).max()
        df['support_50']    = df['low'].rolling(w50).min()
        df['sr_dist_50']    = (c - df['support_50']) / (
            df['resistance_50'] - df['support_50'] + 1e-9
        )
        df['dist_to_resistance_pips'] = df['dist_to_resistance'] * c / PIP_SIZE
        df['dist_to_support_pips']    = df['dist_to_support']    * c / PIP_SIZE
        return df

    def add_volume_profile(self, df):
        df = df.copy()
        c, h, l, v = df['close'], df['high'], df['low'], df['volume']
        tp = (h + l + c) / 3

        vwap_window = min(144, max(5, len(df) // 4))
        obv_window  = min(20,  max(3, len(df) // 10))

        vol_sum    = v.rolling(vwap_window).sum() + 1e-9
        df['vwap'] = (tp * v).rolling(vwap_window).sum() / vol_sum

        df['dist_to_vwap']  = (c - df['vwap']) / (df['vwap'] + 1e-9)
        df['price_vs_vwap'] = (c > df['vwap']).astype(int)

        vwap_std          = (tp - df['vwap']).rolling(vwap_window).std()
        df['vwap_upper1'] = df['vwap'] + 1.0 * vwap_std
        df['vwap_lower1'] = df['vwap'] - 1.0 * vwap_std
        df['vwap_upper2'] = df['vwap'] + 2.0 * vwap_std
        df['vwap_lower2'] = df['vwap'] - 2.0 * vwap_std
        df['vwap_band_pct'] = (c - df['vwap_lower2']) / (
            df['vwap_upper2'] - df['vwap_lower2'] + 1e-9
        )

        obv_dir         = np.sign(c.diff())
        df['obv']       = (v * obv_dir).cumsum()
        df['obv_sma']   = df['obv'].rolling(obv_window).mean()
        df['obv_slope'] = df['obv'].diff(min(5, len(df) // 8)) / (df['obv'].abs() + 1e-9)
        df['obv_vs_sma'] = (df['obv'] > df['obv_sma']).astype(int)
        return df

    def add_digital_indicators(self, df):
        df = df.copy()
        c, h, l = df['close'], df['high'], df['low']
        n = len(df)

        t9  = min(9,  max(3, n // 8))
        t26 = min(26, max(5, n // 5))
        t52 = min(52, max(8, n // 3))
        tenkan   = (h.rolling(t9).max()  + l.rolling(t9).min())  / 2
        kijun    = (h.rolling(t26).max() + l.rolling(t26).min()) / 2
        shift26  = min(26, max(3, n // 6))
        senkou_a = ((tenkan + kijun) / 2).shift(shift26)
        senkou_b = ((h.rolling(t52).max() + l.rolling(t52).min()) / 2).shift(shift26)

        df['ichi_tenkan']         = tenkan
        df['ichi_kijun']          = kijun
        df['ichi_senkou_a']       = senkou_a
        df['ichi_senkou_b']       = senkou_b
        df['ichi_cloud_bull']     = (senkou_a > senkou_b).astype(int)
        df['ichi_above_cloud']    = ((c > senkou_a) & (c > senkou_b)).astype(int)
        df['ichi_tk_cross']       = (
            (tenkan > kijun) & (tenkan.shift(1) <= kijun.shift(1))
        ).astype(int)
        df['ichi_price_vs_kijun'] = (c > kijun).astype(int)

        cmo_p = min(14, max(3, n // 6))
        d_ = c.diff()
        up_ = d_.clip(lower=0).rolling(cmo_p).sum()
        dn_ = (-d_.clip(upper=0)).rolling(cmo_p).sum()
        df['cmo'] = 100 * (up_ - dn_) / (up_ + dn_ + 1e-9)

        er_period = min(10, max(3, n // 8))
        fast_sc, slow_sc = 2/(2+1), 2/(30+1)
        direction  = c.diff(er_period).abs()
        volatility = c.diff().abs().rolling(er_period).sum()
        er = direction / (volatility + 1e-9)
        sc = (er * (fast_sc - slow_sc) + slow_sc) ** 2
        kama_arr = c.to_numpy(dtype=float).copy()
        for i in range(er_period, len(kama_arr)):
            kama_arr[i] = kama_arr[i-1] + sc.iloc[i] * (kama_arr[i] - kama_arr[i-1])
        df['kama']          = kama_arr
        df['price_vs_kama'] = (c > df['kama']).astype(int)
        df['kama_slope']    = pd.Series(kama_arr, index=c.index).diff(min(3, max(1, n//12))) / (c + 1e-9)

        zl_p     = min(21, max(5, n // 5))
        lag_     = (zl_p - 1) // 2
        zl_input = c + (c - c.shift(lag_))
        df['zlema']          = zl_input.ewm(span=zl_p, adjust=False).mean()
        df['price_vs_zlema'] = (c > df['zlema']).astype(int)
        df['zlema_slope']    = df['zlema'].diff(min(3, max(1, n//12))) / (c + 1e-9)

        fd_n = min(20, max(4, n // 5))
        df['fractal_dim'] = (h.rolling(fd_n).max() - l.rolling(fd_n).min()) / (
            df['atr'] * np.sqrt(fd_n) + 1e-9
        )
        return df

    def add_htf_alignment(self, df):
        df = df.copy()
        idx = df.index

        def _g(col):
            return df[col] if col in df.columns else pd.Series(0.0, index=idx)

        trend     = _g('trend_filter')
        ms        = _g('market_structure').clip(-1, 1) * 0.5
        cvd       = _g('cvd_slope').clip(-1, 1)
        vol_div   = _g('volume_divergence')
        adx_v     = (_g('adx') / 100).clip(0, 1)
        ema_align = _g('ema_alignment')
        session   = _g('session_momentum') / 4.0

        df['confirmation_score'] = (
            0.25 * trend
            + 0.20 * ms
            + 0.12 * cvd
            + 0.08 * (1 - vol_div)
            + 0.15 * adx_v
            + 0.10 * ema_align
            + 0.10 * session
        )
        df['signal_strength'] = (
            df['confirmation_score']          * 0.4
            + _g('trend_strength').clip(-1, 1) * 0.3
            + _g('vol_imbalance').clip(-1, 1)  * 0.3
        )
        return df

    # ★ FIX-6: إضافة ميزات regime detection
    def add_regime_features(self, df):
        df = df.copy()
        c = df['close']
        n = len(df)

        # Volatility regime
        p20 = min(20, max(4, n // 5))
        p50 = min(50, max(5, n // 4))
        current_vol = c.pct_change().rolling(p20).std()
        hist_vol    = c.pct_change().rolling(p50).std()
        df['vol_regime_ratio'] = current_vol / (hist_vol + 1e-9)
        # ★ GBP/USD (Cable): تقلب أعلى من EUR/USD/AUD
        # → thresholds أوسع (1.35/0.70 بدل 1.3/0.75)
        df['is_high_vol'] = (df['vol_regime_ratio'] > 1.30).astype(int)  # CHF: عتبة أقل (أهدأ)
        df['is_low_vol']  = (df['vol_regime_ratio'] < 0.72).astype(int)  # CHF: عتبة أعلى

        # Trend strength via ADX buckets
        if 'adx' in df.columns:
            df['adx_strong']  = (df['adx'] > 25).astype(int)
            df['adx_ranging'] = (df['adx'] < 20).astype(int)

        # Distance from recent high/low as percent (mean reversion signal)
        w = min(20, max(5, n // 6))
        df['pct_from_high'] = (df['high'].rolling(w).max() - c) / (c + 1e-9)
        df['pct_from_low']  = (c - df['low'].rolling(w).min())  / (c + 1e-9)

        # RSI divergence
        if 'rsi' in df.columns:
            df['rsi_overbought']  = (df['rsi'] > 70).astype(int)
            df['rsi_oversold']    = (df['rsi'] < 30).astype(int)
            df['rsi_mid_bullish'] = ((df['rsi'] > 50) & (df['rsi'] <= 70)).astype(int)
            df['rsi_mid_bearish'] = ((df['rsi'] < 50) & (df['rsi'] >= 30)).astype(int)

        return df

    def create_features(self, df, shift=True):
        df = self.add_time_features(df)
        df = self.compute_indicators(df)
        df = self.add_advanced_volume(df)
        df = self.add_price_action(df)
        df = self.add_trend_filter(df)
        df = self.add_market_structure(df)
        df = self.add_support_resistance(df)
        df = self.add_volume_profile(df)
        df = self.add_digital_indicators(df)
        df = self.add_htf_alignment(df)
        df = self.add_regime_features(df)   # ★ FIX-6

        if shift:
            exclude_shift = {'datetime','open','high','low','close','volume'}
            shift_cols = [col for col in df.columns if col not in exclude_shift]
            df[shift_cols] = df[shift_cols].shift(1)

        df = df.ffill().fillna(0)
        df.replace([np.inf, -np.inf], 0, inplace=True)

        warmup = max(10, min(60, len(df) // 10))
        df = df.iloc[warmup:].copy()

        thresh = int(len(df) * 0.3)
        df = df.dropna(axis=1, thresh=thresh)
        df = df.ffill().fillna(0)
        df.replace([np.inf, -np.inf], 0, inplace=True)

        exclude_final = {
            'datetime','open','high','low','close','volume',
            'future_close','pct_change','target'
        }
        self.feature_columns = [col for col in df.columns if col not in exclude_final]
        return df

    # ★ FIX-1: create_target مع pip threshold
    def create_target(self, df, horizon=1, pip_threshold=0):
        df = df.copy()

        # أقصى high في الـ horizon شموع القادمة
        df['future_high']  = df['high'].rolling(horizon).max().shift(-horizon)
        df['future_low']   = df['low'].rolling(horizon).min().shift(-horizon)
        df['future_close'] = df['close'].shift(-horizon)

        move_up   = (df['future_high']  - df['close']) / PIP_SIZE
        move_down = (df['close'] - df['future_low'])   / PIP_SIZE

        if pip_threshold > 0:
            # 1 = buy (السعر ارتفع كفاية), 0 = sell/flat
            df['target'] = np.where(
                (move_up > pip_threshold) & (move_up > move_down), 1,
                np.where((move_down > pip_threshold) & (move_down > move_up), 0, -1)
            )
            # حذف الـ neutral (-1) — مش عارفين الاتجاه
            df = df[df['target'] != -1].copy()
        else:
            df['target'] = (df['future_close'] > df['close']).astype(int)

        df['pct_change'] = (df['future_close'] - df['close']) / (df['close'] + 1e-9)
        df.dropna(subset=['future_close'], inplace=True)
        df['target'] = df['target'].astype(int)
        return df

    def fit_scaler(self, X):
        self.scaler.fit(X)
        self.scaler_fitted = True
        if isinstance(X, pd.DataFrame):
            self.feature_columns = list(X.columns)

    def transform(self, X):
        if not self.scaler_fitted:
            raise RuntimeError("يجب تدريب الـ scaler أولاً")
        if isinstance(X, pd.DataFrame):
            for col in self.feature_columns:
                if col not in X.columns: X[col] = 0.0
            X = X[self.feature_columns]
        Xs = self.scaler.transform(X)
        # ★ V2 BUG FIX: بعد ما feature selection شغّال، لازم transform
        # يرجع فقط الفيتشورز المُختارة، مش كل الـ 220. غير كده الموديل
        # المُدرَّب على 45 هيرمي Feature shape mismatch errors عند التنبؤ.
        if self.selected_cols is not None and len(self.selected_cols) < len(self.feature_columns):
            feat_idx = [self.feature_columns.index(c) for c in self.selected_cols
                        if c in self.feature_columns]
            Xs = Xs[:, feat_idx]
        return Xs

    def fit_transform(self, X):
        self.fit_scaler(X)
        return self.transform(X)

    def fit_feature_selection(self, X_scaled, y):
        """
        ★ V2 NEW: Feature pruning للحد من overfitting.
        الدالة دي مش موجودة في V1 من Technical model.
        منطق الاختيار:
          - top-K صارم حسب importance
          - شرط importance > 0 (نتجاهل الفيتشورز اللي XGBoost مالمسهاش)
          - حد أقصى حسب حجم العينة: 1 feature لكل 100 sample
        """
        n_samples = len(y)
        if n_samples < 300:
            logging.info(f"  ⚠️  عينة صغيرة ({n_samples}) — تخطي feature selection")
            self.selected_cols = list(self.feature_columns)
            return self.selected_cols

        logging.info(f"  Feature Selection on {X_scaled.shape[1]} features...")
        # ★ V7 FIX: Balance class weight for feature selection XGB
        neg_c = int((y == 0).sum())
        pos_c = int((y == 1).sum())
        spw = (neg_c / pos_c) if pos_c > 0 else 1.0
        # Cap spw to avoid extreme imbalance skewing feature importance
        spw = max(0.3, min(spw, 3.0))

        sel_model = xgb.XGBClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.7,
            scale_pos_weight=spw,   # ★ V7 NEW
            random_state=42, tree_method='hist',
            verbosity=0, use_label_encoder=False
        )
        sel_model.fit(X_scaled, y)
        importances = sel_model.feature_importances_

        max_by_sample = max(15, n_samples // 100)
        max_features = min(45, max_by_sample, len(importances))

        sorted_indices = np.argsort(importances)[::-1]
        selected_indices = []
        for idx in sorted_indices:
            if importances[idx] > 0 and len(selected_indices) < max_features:
                selected_indices.append(int(idx))

        if len(selected_indices) < 10:
            selected_indices = [int(i) for i in sorted_indices[:max(10, min(15, len(importances)))]]

        self.selected_cols = [self.feature_columns[i] for i in selected_indices
                               if i < len(self.feature_columns)]
        logging.info(
            f"  ✂️ Feature pruning: {len(self.feature_columns)} → "
            f"{len(self.selected_cols)} features "
            f"(samples={n_samples}, ratio={n_samples/max(len(self.selected_cols),1):.0f}/feature)"
        )
        return self.selected_cols


# ══════════════════════════════════════════════════════════════
# MULTI-TIMEFRAME BUILDER
# ══════════════════════════════════════════════════════════════
class MultiTimeframeBuilder:
    def __init__(self):
        self._htf_cache = {}

    def _build_htf_df(self, df_raw, resample_rule):
        if resample_rule in self._htf_cache:
            return self._htf_cache[resample_rule]
        df_htf = (
            df_raw.set_index('datetime')
            .resample(resample_rule)
            .agg({'open':'first','high':'max','low':'min','close':'last','volume':'sum'})
            .dropna().reset_index()
        )
        fe_htf   = FeatureEngineer()
        df_htf_f = fe_htf.create_features(df_htf, shift=True)
        df_htf_f = fe_htf.create_target(df_htf_f, horizon=1)
        self._htf_cache[resample_rule] = df_htf_f
        return df_htf_f

    def merge_htf_features(self, df_ltf, df_raw, htf_resample_rule,
                           htf_feature_names, htf_label):
        if not htf_feature_names or htf_resample_rule is None:
            return df_ltf
        df_htf    = self._build_htf_df(df_raw, htf_resample_rule)
        available = [c for c in htf_feature_names if c in df_htf.columns]
        if not available: return df_ltf
        df_sel = df_htf[['datetime'] + available].copy()
        df_sel.columns = ['datetime'] + [f'htf_{htf_label}_{c}' for c in available]
        df_merged = pd.merge_asof(
            df_ltf.sort_values('datetime'),
            df_sel.sort_values('datetime'),
            on='datetime', direction='backward'
        )
        return df_merged.ffill().fillna(0)


# ══════════════════════════════════════════════════════════════
# BACKTESTER — محسّن لـ EUR/USD
# ★ FIX-7: إضافة trailing stop + minimum pip move للدخول
# ══════════════════════════════════════════════════════════════
class Backtester:
    def __init__(self, initial_capital=10000, spread_bps=SPREAD_BPS, risk_pct=0.02):
        # ★ GBP/USD: spread ~1.5 bps، risk_pct موحّد
        self.initial_capital = initial_capital
        self.spread_bps      = spread_bps / 10000
        self.slippage        = 0.00005
        self.risk_pct        = risk_pct

    def _exec_price(self, price, is_buy):
        if is_buy:
            return price * (1 + self.spread_bps) * (1 + self.slippage)
        return price * (1 - self.spread_bps) * (1 - self.slippage)

    def run(self, df, model, fe, feature_columns, timeframe_config,
            atr_mult_stop=ATR_MULT_STOP, atr_mult_take=ATR_MULT_TAKE,
            use_fold_thresholds=False):

        capital   = self.initial_capital
        positions = []
        trades    = []
        equity    = [capital]

        a_stop = timeframe_config.get('atr_mult_stop', atr_mult_stop)
        a_take = timeframe_config.get('atr_mult_take', atr_mult_take)

        missing = [c for c in feature_columns + ['close','atr']
                   if c not in df.columns]
        if missing:
            logging.error(f"أعمدة مفقودة: {missing}")
            return {}

        key_buy  = 'fold_buy_threshold'  if use_fold_thresholds else 'buy_threshold'
        key_sell = 'fold_sell_threshold' if use_fold_thresholds else 'sell_threshold'
        buy_th   = timeframe_config.get(key_buy,  0.57)
        sell_th  = timeframe_config.get(key_sell, 0.43)

        # ★ V7 FIX: Adaptive Threshold Mode
        # المشكلة: بعض الفريمات (15min خاصة) بتنتج توقعات كلها حوالين 0.50
        # فأي عتبة ثابتة > 0.55 بترجع صفر صفقات.
        # الحل: لو مُفعّل، استخدم percentile من التوقعات الفعلية بدل عتبة ثابتة.
        use_adaptive = timeframe_config.get('use_adaptive_threshold', False)
        adaptive_top_pct = timeframe_config.get('adaptive_top_pct', 15)  # top 15% = buy

        max_daily    = timeframe_config.get('max_trades_per_day')
        daily_counts = {}

        try:
            X_scaled  = fe.transform(df[feature_columns].copy())

            # ★ V2 SAFETY NET: تأكيد إضافي — لو model.n_features_in_ مش
            # متطابق مع X_scaled.shape[1]، الفلتر لم يُنفَّذ صح. نحاول
            # استخراج أول N features (top-K by importance في training).
            if hasattr(model, 'n_features_in_') and X_scaled.shape[1] != model.n_features_in_:
                expected = model.n_features_in_
                got      = X_scaled.shape[1]
                logging.warning(
                    f"Feature count mismatch in Backtester: "
                    f"model expects {expected} but got {got}. "
                    f"Attempting emergency filter to first {expected} columns."
                )
                if got > expected:
                    X_scaled = X_scaled[:, :expected]
                else:
                    logging.error(
                        f"Can't recover: got {got} features but need {expected}. "
                        f"Model file may be from a different version."
                    )
                    return {}
            all_proba = model.predict_proba(X_scaled)[:, 1]

            # ★ V7: Apply adaptive threshold if enabled
            if use_adaptive and len(all_proba) > 50:
                # Top adaptive_top_pct% → buy signal
                buy_th_dyn = float(np.percentile(all_proba, 100 - adaptive_top_pct))
                sell_th_dyn = float(np.percentile(all_proba, adaptive_top_pct))
                # Only override if the dynamic threshold is looser than static
                # (never make static threshold more restrictive than user configured)
                if buy_th_dyn < buy_th:
                    logging.info(f"  V7 Adaptive: buy_th {buy_th:.3f} → {buy_th_dyn:.3f} "
                                 f"(top {adaptive_top_pct}%)")
                    buy_th = buy_th_dyn
                if sell_th_dyn > sell_th:
                    logging.info(f"  V7 Adaptive: sell_th {sell_th:.3f} → {sell_th_dyn:.3f}")
                    sell_th = sell_th_dyn
        except Exception as e:
            logging.error(f"Predict error: {e}")
            return {}
        closes    = df['close'].values
        atrs      = df['atr'].values
        datetimes = df['datetime'].values if 'datetime' in df.columns else [None] * len(df)

        # ★ V8: Extract HTF alignment + ADX filter arrays if needed
        require_htf = timeframe_config.get('require_htf_alignment', False)
        adx_gate = timeframe_config.get('adx_min_entry', 0)
        htf_trend_arr = df['trend_filter'].values if ('trend_filter' in df.columns and require_htf) else None
        adx_arr = df['adx'].values if ('adx' in df.columns and adx_gate > 0) else None

        for i in range(len(df)):
            p, atr, prob = closes[i], max(float(atrs[i]), 1e-6), all_proba[i]

            signal = "HOLD"
            if prob > buy_th:  signal = "BUY"
            elif prob < sell_th: signal = "SELL"

            # ★ V8: HTF Alignment Gate — الإشارة لازم في نفس اتجاه HTF trend
            # trend_filter: >0 = uptrend, <0 = downtrend
            if require_htf and htf_trend_arr is not None and signal != "HOLD":
                htf_val = float(htf_trend_arr[i]) if not np.isnan(htf_trend_arr[i]) else 0
                if signal == "BUY" and htf_val <= 0:
                    signal = "HOLD"   # buy لكن HTF لا يدعم
                elif signal == "SELL" and htf_val >= 0:
                    signal = "HOLD"

            # ★ V8: ADX Gate — الإشارة لازم في trend قوي
            if adx_gate > 0 and adx_arr is not None and signal != "HOLD":
                adx_val = float(adx_arr[i]) if not np.isnan(adx_arr[i]) else 0
                if adx_val < adx_gate:
                    signal = "HOLD"

            if signal == "BUY" and max_daily and datetimes[i] is not None:
                dk = str(datetimes[i])[:10]
                if daily_counts.get(dk, 0) >= max_daily: signal = "HOLD"

            if signal == "BUY" and not positions:
                ep    = self._exec_price(p, True)
                units = capital * self.risk_pct / ep
                positions.append({
                    'entry_price': ep, 'units': units,
                    'stop_loss':   ep - a_stop * atr,
                    'take_profit': ep + a_take * atr,
                    'peak_price':  ep   # ★ للـ trailing stop
                })
                capital -= units * ep
                if max_daily and datetimes[i] is not None:
                    dk = str(datetimes[i])[:10]
                    daily_counts[dk] = daily_counts.get(dk, 0) + 1

            elif signal == "SELL" and positions:
                pos = positions.pop(0)
                ep  = self._exec_price(p, False)
                pnl = pos['units'] * (ep - pos['entry_price'])
                capital += pos['units'] * ep
                trades.append({
                    'profit': pnl,
                    'return_pct': pnl / (pos['units'] * pos['entry_price'] + 1e-9),
                    'exit_reason': 'signal'
                })

            if positions:
                pos = positions[0]
                # ★ FIX-7: trailing stop — بنتبع الـ peak
                if p > pos['peak_price']:
                    pos['peak_price'] = p
                    trail_stop = p - a_stop * atr
                    if trail_stop > pos['stop_loss']:
                        pos['stop_loss'] = trail_stop

                if p <= pos['stop_loss']:
                    ep  = pos['stop_loss']
                    pnl = pos['units'] * (ep - pos['entry_price'])
                    capital += pos['units'] * ep
                    trades.append({
                        'profit': pnl,
                        'return_pct': pnl / (pos['units'] * pos['entry_price'] + 1e-9),
                        'exit_reason': 'stop_loss'
                    })
                    positions.pop(0)
                elif p >= pos['take_profit']:
                    ep  = pos['take_profit']
                    pnl = pos['units'] * (ep - pos['entry_price'])
                    capital += pos['units'] * ep
                    trades.append({
                        'profit': pnl,
                        'return_pct': pnl / (pos['units'] * pos['entry_price'] + 1e-9),
                        'exit_reason': 'take_profit'
                    })
                    positions.pop(0)

            mtm = positions[0]['units'] * p if positions else 0
            equity.append(capital + mtm)

        if positions:
            pos = positions.pop(0)
            ep  = self._exec_price(closes[-1], False)
            pnl = pos['units'] * (ep - pos['entry_price'])
            capital += pos['units'] * ep
            trades.append({
                'profit': pnl,
                'return_pct': pnl / (pos['units'] * pos['entry_price'] + 1e-9),
                'exit_reason': 'end_of_data'
            })

        if not trades: return {}

        eq   = pd.Series(equity)
        wins = [t for t in trades if t['profit'] > 0]
        loss = [t for t in trades if t['profit'] <= 0]

        # ★ V8 FIX: Outlier cap يُطبَّق على capital + return + PF جميعاً
        # في V7 كان بيصلح PF بس، capital/return لسه بيعكس الـ outlier
        # الحل: نعيد حساب capital من trades المُقلَّمة
        outlier_cap = timeframe_config.get('outlier_cap_multiplier', None)
        capital_adjustment = 0.0
        if outlier_cap and wins and len(wins) >= 5:
            profits_sorted = sorted([t['profit'] for t in wins])
            median_win = profits_sorted[len(profits_sorted) // 2]
            cap_val = median_win * outlier_cap
            n_capped = 0
            for w in wins:
                if w['profit'] > cap_val:
                    capital_adjustment -= (w['profit'] - cap_val)   # ★ V8: نتتبع الفارق
                    w['profit'] = cap_val
                    n_capped += 1
            if n_capped > 0:
                logging.info(f"  V8 Outlier cap: {n_capped} صفقة اتـcapped عند {cap_val:.2f} "
                             f"(capital adjust: {capital_adjustment:.2f})")
                # ★ V8: طبق التعديل على capital
                capital += capital_adjustment
                # وأعد بناء الـ equity curve بنفس النسبة
                if capital > 0:
                    scale = 1.0 - abs(capital_adjustment) / max(self.initial_capital, 1)
                    if scale > 0.1:
                        equity = [e * scale + self.initial_capital * (1 - scale) for e in equity]

        wr   = len(wins) / len(trades) * 100 if trades else 0
        tp   = sum(t['profit'] for t in wins)
        tl   = abs(sum(t['profit'] for t in loss))
        dd   = ((eq - eq.cummax()) / (eq.cummax() + 1e-9) * 100).min()
        aw   = tp / len(wins) if wins else 0
        al   = tl / len(loss) if loss else 0
        pf   = (tp / tl) if tl > 0 else float('inf')

        # ★ V8 FIX #4: Reliability flags للـ metrics
        # is_reliable = صفقات كافية + WR منطقي + PF غير مبالغ
        # ★ V8 NEW: min_reliable_trades override من الـ config (1W خصوصاً)
        min_trades_required = timeframe_config.get('min_reliable_trades', 15)
        is_reliable = (
            len(trades) >= min_trades_required
            and 20 <= wr <= 90
            and pf < 10
            and pf > 0
            and not (pf == float('inf'))
        )
        # PF outlier detection — لو أعلى من 10، فحص يدوي مطلوب
        pf_suspicious = pf > 10 and len(trades) < 30
        # avg_win/avg_loss ratio للفحص
        implied_rr = (pf * (1 - wr/100) / (wr/100)) if wr > 0 else 0

        return {
            'total_trades':     len(trades),
            'win_rate':         round(wr, 2),
            'profit_factor':    round(pf, 2) if pf != float('inf') else 99.99,
            'total_return_pct': round((capital - self.initial_capital) / self.initial_capital * 100, 2),
            'max_drawdown_pct': round(dd, 2),
            'avg_win':          round(aw, 2),
            'avg_loss':         round(al, 2),
            'expectancy':       round((wr / 100 * aw) - ((1 - wr / 100) * al), 2) if al else 0,
            'is_reliable':      bool(is_reliable),        # ★ V7 NEW
            'pf_suspicious':    bool(pf_suspicious),      # ★ V7 NEW
            'implied_win_loss_ratio': round(implied_rr, 2) # ★ V7 NEW
        }


# ══════════════════════════════════════════════════════════════
# HELPER: Single Fold
# ══════════════════════════════════════════════════════════════
def _train_single_fold(X, y, xgb_param, timeframe):
    neg = (y == 0).sum()
    pos = (y == 1).sum()
    spw = neg / pos if pos > 0 else 1.0
    n_est = xgb_param.pop('n_estimators', 800)

    fe_fold = FeatureEngineer()
    fe_fold.fit_scaler(X)
    X_s = fe_fold.transform(X)

    model = xgb.XGBClassifier(
        objective='binary:logistic',
        n_estimators=n_est,
        scale_pos_weight=spw,
        random_state=42, tree_method='hist',
        eval_metric='logloss', verbosity=0,
        **xgb_param
    )
    model.fit(X_s, y, verbose=False)
    xgb_param['n_estimators'] = n_est
    logging.info(f"[{timeframe}] ✅ Single-fold — {len(X)} عينة")
    return model, fe_fold


# ══════════════════════════════════════════════════════════════
# WALK-FORWARD TRAINING
# ══════════════════════════════════════════════════════════════
def train_model_with_walk_forward(df_raw, timeframe, n_splits=5, mtf_builder=None):
    config        = TIMEFRAME_CONFIG[timeframe]
    resample_rule = config['resample_rule']
    years         = config['years']
    min_candles   = config.get('min_candles', 100)
    larger_tf     = config.get('larger_tf')
    htf_features  = config.get('htf_features', [])
    horizon       = HORIZON.get(timeframe, 1)
    pip_thr       = PIP_THRESHOLD.get(timeframe, 0)

    logging.info("=" * 60)
    logging.info(f"Walk-Forward [USD/CHF] — {timeframe} | horizon={horizon} | pip_thr={pip_thr}")
    logging.info("=" * 60)

    df_tf = (
        df_raw.set_index('datetime')
        .resample(resample_rule)
        .agg({'open':'first','high':'max','low':'min','close':'last','volume':'sum'})
        .dropna().reset_index()
    )
    if 'datetime' not in df_tf.columns:
        df_tf.rename(columns={'index':'datetime'}, inplace=True)

    if years:
        cutoff = df_tf['datetime'].max() - timedelta(days=365 * years)
        df_tf  = df_tf[df_tf['datetime'] >= cutoff].copy()

    logging.info(f"شموع بعد القطع: {len(df_tf)}")

    if len(df_tf) < min_candles:
        logging.error(f"❌ {timeframe}: {len(df_tf)} < {min_candles}")
        return None

    fe        = FeatureEngineer()
    df_feat   = fe.create_features(df_tf, shift=True)

    # ★ FIX-1: استخدام pip_threshold في create_target
    df_target = fe.create_target(df_feat, horizon=horizon, pip_threshold=pip_thr)

    if df_target.empty or len(df_target) < min_candles:
        logging.error(f"❌ {timeframe}: بيانات غير كافية بعد target ({len(df_target)})")
        return None

    # class balance check
    class_counts = df_target['target'].value_counts()
    logging.info(f"Class balance: {class_counts.to_dict()}")
    if len(class_counts) < 2:
        logging.error(f"❌ {timeframe}: class واحدة فقط بعد الـ pip threshold")
        return None

    if mtf_builder and larger_tf and htf_features:
        htf_cfg = TIMEFRAME_CONFIG.get(larger_tf)
        if htf_cfg:
            df_target = mtf_builder.merge_htf_features(
                df_ltf=df_target, df_raw=df_raw,
                htf_resample_rule=htf_cfg['resample_rule'],
                htf_feature_names=htf_features,
                htf_label=larger_tf
            )

    exclude_final = {
        'datetime','open','high','low','close','volume',
        'future_close','future_high','future_low','pct_change','target'
    }
    fe.feature_columns = [c for c in df_target.columns if c not in exclude_final]
    logging.info(f"Features: {len(fe.feature_columns)}")

    X = df_target[fe.feature_columns]
    y = df_target['target']

    xgb_param = dict(XGB_PARAMS.get(timeframe, XGB_PARAMS["default"]))
    n_est_cfg = xgb_param.pop('n_estimators', 800)

    if len(X) < 100:
        min_train = max(15, len(X) // 4)
        min_test  = max(5,  len(X) // 10)
        n_splits  = min(n_splits, 3)
    else:
        min_train = max(30, len(X) // (n_splits + 1))
        min_test  = max(8,  len(X) // (n_splits * 3))

    effective_splits = min(n_splits, max(2, len(X) // (min_train + min_test)))
    tscv = TimeSeriesSplit(n_splits=effective_splits)

    fold_metrics = []
    best_model   = None
    best_fe_fold = None
    best_score   = -np.inf
    last_model   = None
    last_fe_fold = None

    for fold, (tr_idx, te_idx) in enumerate(tscv.split(X)):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

        if len(X_tr) < min_train or len(X_te) < min_test:
            logging.warning(f"Fold {fold+1}: train={len(X_tr)} test={len(X_te)} — تخطي")
            continue

        neg = (y_tr == 0).sum()
        pos = (y_tr == 1).sum()
        if pos == 0 or neg == 0:
            logging.warning(f"Fold {fold+1}: class واحدة — تخطي")
            continue
        spw = neg / pos

        fe_fold = FeatureEngineer()
        fe_fold.fit_scaler(X_tr)
        X_tr_s = fe_fold.transform(X_tr)
        X_te_s = fe_fold.transform(X_te)

        # ★ V2 NEW: Apply feature selection in CV too (consistency with production)
        # قبل: كان CV يستخدم كل 220 فيتشور، وProduction فقط بيستخدم feature
        # selection. النتيجة: CV metrics مش بتعكس أداء production الحقيقي.
        # دلوقتي: نفس الفلتر يُطبَّق في CV و production.
        use_fs_cv = config.get('use_feature_selection', True)
        if use_fs_cv:
            fe_fold.fit_feature_selection(X_tr_s, y_tr)
            if fe_fold.selected_cols and len(fe_fold.selected_cols) < len(fe_fold.feature_columns):
                feat_idx = [fe_fold.feature_columns.index(c) for c in fe_fold.selected_cols
                            if c in fe_fold.feature_columns]
                X_tr_s = X_tr_s[:, feat_idx]
                X_te_s = X_te_s[:, feat_idx]

        # ★ FIX-4: n_estimators من الـ config + early stopping
        model = xgb.XGBClassifier(
            objective='binary:logistic',
            n_estimators=n_est_cfg,
            early_stopping_rounds=80,
            scale_pos_weight=spw,
            random_state=42, tree_method='hist',
            eval_metric='logloss', verbosity=0,
            **xgb_param
        )
        model.fit(X_tr_s, y_tr, eval_set=[(X_te_s, y_te)], verbose=False)

        last_model   = model
        last_fe_fold = fe_fold

        y_pred    = model.predict(X_te_s)
        acc       = accuracy_score(y_te, y_pred)
        precision = precision_score(y_te, y_pred, zero_division=0)
        recall    = recall_score(y_te, y_pred, zero_division=0)
        f1        = f1_score(y_te, y_pred, zero_division=0)

        df_test      = df_target.iloc[te_idx].copy()
        backtester   = Backtester()
        metrics_test = backtester.run(
            df_test, model, fe_fold,
            fe_fold.feature_columns, config,
            use_fold_thresholds=True
        )

        logging.info(
            f"  Fold {fold+1} — Acc:{acc:.2%}  F1:{f1:.2%}  "
            f"Prec:{precision:.2%}  Rec:{recall:.2%}"
        )
        if metrics_test:
            logging.info(
                f"  BT — WR:{metrics_test['win_rate']}%  "
                f"PF:{metrics_test['profit_factor']}  "
                f"Ret:{metrics_test['total_return_pct']}%  "
                f"Trades:{metrics_test['total_trades']}"
            )

        fold_metrics.append({
            'fold': fold+1, 'accuracy': acc, 'f1': f1,
            'precision': precision, 'recall': recall,
            'backtest': metrics_test
        })

        # Score مركب
        if metrics_test and metrics_test.get('total_trades', 0) >= 3:
            score = (
                metrics_test.get('profit_factor', 0) * 0.45
                + metrics_test.get('win_rate',    0) * 0.20
                + min(metrics_test.get('total_trades', 0), 50) * 0.25
                + f1 * 100                                      * 0.10
            )
        else:
            score = f1 * 100

        if score > best_score:
            best_score   = score
            best_model   = model
            best_fe_fold = fe_fold

    if best_model is None:
        logging.warning(f"⚠️  [{timeframe}] → Single Fold Training")
        xgb_param['n_estimators'] = n_est_cfg
        best_model, best_fe_fold = _train_single_fold(X, y, xgb_param, timeframe)
        xgb_param.pop('n_estimators', None)

    # Final model على كل البيانات
    neg_f = (y == 0).sum()
    pos_f = (y == 1).sum()
    n_est_final = n_est_cfg
    if hasattr(best_model, 'best_iteration') and best_model.best_iteration:
        n_est_final = max(100, best_model.best_iteration + 1)

    fe_full = FeatureEngineer()
    fe_full.fit_scaler(X)
    X_full_s = fe_full.transform(X)

    # ★ V2 NEW: Feature Selection — تقلل خطر overfitting
    use_fs = config.get('use_feature_selection', True)
    if use_fs:
        fe_full.fit_feature_selection(X_full_s, y)
        if fe_full.selected_cols and len(fe_full.selected_cols) < len(fe_full.feature_columns):
            feat_idx = [fe_full.feature_columns.index(c) for c in fe_full.selected_cols
                        if c in fe_full.feature_columns]
            X_full_s = X_full_s[:, feat_idx]

    final_model = xgb.XGBClassifier(
        objective='binary:logistic',
        n_estimators=n_est_final,
        scale_pos_weight=neg_f / pos_f if pos_f else 1,
        random_state=42, tree_method='hist',
        eval_metric='logloss', verbosity=0,
        **xgb_param
    )
    final_model.fit(X_full_s, y)

    os.makedirs(MODELS_DIR, exist_ok=True)
    model_path = f"{MODELS_DIR}/{SYMBOL_SAFE}_{timeframe}.pkl"
    joblib.dump({
        'model':           final_model,
        'scaler':          fe_full.scaler,
        'feature_columns': fe_full.feature_columns,
        'selected_cols':   fe_full.selected_cols,   # ★ V2 NEW
        'config':          config,
        'horizon':         horizon,
        'pip_threshold':   pip_thr
    }, model_path)

    holdout_start = int(len(df_target) * 0.8)
    df_holdout    = df_target.iloc[holdout_start:].copy()
    full_metrics  = {}

    if len(df_holdout) >= 10:
        bt_holdout   = Backtester()
        full_metrics = bt_holdout.run(
            df_holdout, final_model, fe_full,
            fe_full.feature_columns, config
        )
        if full_metrics:
            logging.info(
                f"✅ Holdout — WR:{full_metrics['win_rate']}%  "
                f"PF:{full_metrics['profit_factor']}  "
                f"Ret:{full_metrics['total_return_pct']}%  "
                f"Trades:{full_metrics['total_trades']}"
            )

    metrics_file = f"{MODELS_DIR}/{SYMBOL_SAFE}_{timeframe}_metrics.json"
    with open(metrics_file, 'w', encoding='utf-8') as f:
        json.dump({
            'symbol': SYMBOL, 'timeframe': timeframe,
            'n_features': len(fe_full.feature_columns),
            'selected_features': len(fe_full.selected_cols) if fe_full.selected_cols else len(fe_full.feature_columns),  # ★ V2
            'horizon': horizon, 'pip_threshold': pip_thr,
            'fold_results': fold_metrics,
            'holdout_backtest_last_20pct': full_metrics,
            'config': config
        }, f, indent=2, default=str, ensure_ascii=False)

    logging.info(f"💾 {model_path}")
    return final_model, fe_full


# ══════════════════════════════════════════════════════════════
# TRAIN ALL MODELS
# ══════════════════════════════════════════════════════════════
# ★ V2 NEW: تحديد الفريمات النشطة (بدل تدريب كل الـ 8)
# مبني على تحليل CV من Classic model — الفريمات الوسطى (15min-2h) الأقوى
ACTIVE_TIMEFRAMES_V2 = ['5min', '15min', '45min', '1h', '2h', '4h', '1d', '1W']  # ★ V5: كل الـ 8
# ملاحظة: 5min, 4h, 1d, 1W معطّلين احتياطياً.
# لو الـ Technical عندك أعطى نتايج مختلفة عن Classic، عدّل القائمة دي.


def train_all_models(force_refresh_data=False):
    logging.info("=" * 70)
    logging.info("★ USD/CHF (Swissy) Technical Model Factory — Europe+US ★")
    logging.info("=" * 70)

    df_5min = get_or_update_data(SYMBOL_SAFE, interval="5min", force_refresh=force_refresh_data)
    df_1h   = get_or_update_data(SYMBOL_SAFE, interval="1h",   force_refresh=force_refresh_data)

    mtf_builder = MultiTimeframeBuilder()
    results     = {}

    for tf in TIMEFRAME_CONFIG:
        # ★ V2: تخطي الفريمات غير النشطة (بدل تدريب 8 فريمات بلا داعي)
        if tf not in ACTIVE_TIMEFRAMES_V2:
            logging.info(f"⏭️  تخطي {tf} — غير نشط في V2 (راجع ACTIVE_TIMEFRAMES_V2)")
            results[tf] = "⏭️ معطّل"
            continue
        logging.info("-" * 50)
        try:
            source_df = df_1h if tf in ["1h","2h","4h","1d","1W"] else df_5min
            if source_df is None or source_df.empty:
                results[tf] = "❌ فشل البيانات"
                continue
            res = train_model_with_walk_forward(source_df, tf, n_splits=5, mtf_builder=mtf_builder)
            results[tf] = "✅ نجح" if res else "❌ فشل"
        except Exception as e:
            logging.error(f"❌ {tf}: {e}", exc_info=True)
            results[tf] = f"❌ خطأ: {e}"

    logging.info("=" * 70)
    logging.info("ملخص:")
    for tf, st in results.items():
        logging.info(f"   {tf:6s}: {st}")
    logging.info(f"📁 {MODELS_DIR}/")


if __name__ == "__main__":
    train_all_models(force_refresh_data=True)

## — عرض النتايج

In [9]:
import os
import json
import pandas as pd
from pathlib import Path
import numpy as np

# ──────────────────────────────────────────────
# إعدادات الموديل التقني V7
# ──────────────────────────────────────────────
MODELS_DIR = "/content/models/usdchf_technical_v8"
SYMBOL     = "USDCHF"

TIMEFRAMES = ["5min", "15min", "45min", "1h", "2h", "4h", "1d", "1W"]

def load_metrics(tf: str):
    path = Path(MODELS_DIR) / f"{SYMBOL}_{tf}_metrics.json"
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def safe_mean(values, default=None):
    vals = [v for v in values if v is not None and not (isinstance(v, float) and (np.isinf(v) or np.isnan(v)))]
    return float(np.mean(vals)) if vals else default

def main():
    rows = []

    for tf in TIMEFRAMES:
        m = load_metrics(tf)
        if m is None:
            rows.append({
                "TF": tf,
                "Folds": "—",
                "Acc": "—",
                "F1": "—",
                "WR%": "—",
                "PF": "—",
                "Trades": "—",
                "Holdout WR": "—",
                "Holdout PF": "—",
                "Holdout Trades": "—",
                "Reliable": "—",
                "Features": "—",
                "Status": "❌ Missing"
            })
            continue

        folds = m.get("fold_results", [])
        ho    = m.get("holdout_backtest_last_20pct", {}) or {}

        accs, f1s, wrs, pfs, trades_list = [], [], [], [], []

        for f in folds:
            if "accuracy" in f:
                accs.append(f["accuracy"])
            if "f1" in f:
                f1s.append(f["f1"])

            bt = f.get("backtest") or {}
            if bt.get("total_trades", 0) > 0:
                wrs.append(bt.get("win_rate"))
                pf_val = bt.get("profit_factor")
                if pf_val is not None and not (isinstance(pf_val, float) and np.isinf(pf_val)):
                    pfs.append(pf_val)
                trades_list.append(bt.get("total_trades"))

        avg_acc    = safe_mean(accs)
        avg_f1     = safe_mean(f1s)
        avg_wr     = safe_mean(wrs)
        avg_pf     = safe_mean(pfs)
        avg_trades = safe_mean(trades_list)

        # V7 reliability flags from holdout
        is_reliable = ho.get("is_reliable", None)
        pf_suspicious = ho.get("pf_suspicious", None)

        reliable_str = "—"
        if is_reliable is True:
            reliable_str = "✅"
        elif is_reliable is False:
            reliable_str = "⚠️"
        if pf_suspicious:
            reliable_str = "🚨 Suspicious"

        def fmt(val, decimals=2, pct=False):
            if val is None:
                return "—"
            if pct:
                return f"{val*100:.{decimals}f}%"
            return f"{val:.{decimals}f}"

        ho_pf = ho.get("profit_factor")
        if isinstance(ho_pf, float) and np.isinf(ho_pf):
            ho_pf_str = "inf"
        else:
            ho_pf_str = fmt(ho_pf, 2)

        rows.append({
            "TF":             tf,
            "Folds":          len(folds),
            "Acc":            fmt(avg_acc, 1, pct=True),
            "F1":             fmt(avg_f1, 3),
            "WR%":            fmt(avg_wr, 1),
            "PF":             fmt(avg_pf, 2),
            "Trades":         fmt(avg_trades, 0),
            "Holdout WR":     f"{ho.get('win_rate', 0):.1f}%" if isinstance(ho.get("win_rate"), (int, float)) else "—",
            "Holdout PF":     ho_pf_str,
            "Holdout Trades": ho.get("total_trades", "—"),
            "Reliable":       reliable_str,
            "Features":       f"{m.get('selected_features', '—')}/{m.get('n_features', '—')}",
            "Status":         "✅ OK"
        })

    df = pd.DataFrame(rows)

    print("\n" + "=" * 120)
    print("  USD/CHF Technical Model — Results Summary (V7)")
    print("=" * 120)
    print(df.to_string(index=False))
    print("=" * 120)

    # ملاحظات سريعة
    print("\n📌 ملاحظات سريعة:")
    for r in rows:
        if r["Status"] != "✅ OK":
            print(f"  • {r['TF']}: الملف مش موجود")
            continue

        notes = []
        try:
            pf = float(r["PF"]) if r["PF"] not in ("—", "inf") else None
            wr = float(r["WR%"]) if r["WR%"] != "—" else None
            trades = float(r["Trades"]) if r["Trades"] != "—" else None
            ho_trades = r["Holdout Trades"]

            if pf is not None and pf < 1.0:
                notes.append("PF < 1 (خاسر)")
            if pf is not None and pf > 5.0:
                notes.append(f"PF مرتفع ({pf:.1f})")
            if wr is not None and wr < 40:
                notes.append(f"WR منخفض ({wr:.0f}%)")
            if trades is not None and trades < 15:
                notes.append(f"صفقات قليلة ({trades:.0f})")
            if isinstance(ho_trades, (int, float)) and ho_trades < 15:
                notes.append(f"Holdout صفقات قليلة ({ho_trades})")
            if r["Reliable"] == "🚨 Suspicious":
                notes.append("PF مشبوه (outlier)")
            elif r["Reliable"] == "⚠️":
                notes.append("غير موثوق إحصائياً")
        except:
            pass

        if notes:
            print(f"  • {r['TF']}: {' | '.join(notes)}")
        else:
            print(f"  • {r['TF']}: أداء مقبول")

if __name__ == "__main__":
    main()


  USD/CHF Technical Model — Results Summary (V7)
   TF  Folds   Acc    F1  WR%   PF Trades Holdout WR Holdout PF  Holdout Trades     Reliable Features Status
 5min      4 52.1% 0.511 30.8 0.60     55      52.2%       2.62              69            ✅   45/178   ✅ OK
15min      4 53.2% 0.533 42.6 1.13     20      60.7%       2.22              28            ✅   15/177   ✅ OK
45min      4 55.5% 0.472 57.3 2.96     12     100.0%      99.99              15 🚨 Suspicious   15/176   ✅ OK
   1h      4 54.5% 0.603 45.4 1.26    249      60.3%       2.56             146            ✅   45/177   ✅ OK
   2h      4 54.7% 0.579 48.7 1.44    107      63.3%       3.34              98            ✅   37/176   ✅ OK
   4h      4 53.4% 0.556 53.1 2.06     48      75.0%       4.70              40            ✅   20/177   ✅ OK
   1d      4 52.5% 0.518 43.0 1.11     13      75.0%       5.50              24            ✅   15/175   ✅ OK
   1W      4 55.5% 0.521 50.0 2.40      2     100.0%      99.99               

## — Download Models

In [10]:
# Download V5 Technical models
import os, shutil
from google.colab import files

MODELS_DIR = "models/usdchf_technical_v8"
ZIP_NAME   = "usdchf_technical_v8.zip"

if not os.path.exists(MODELS_DIR):
    print(f"❌ '{MODELS_DIR}' غير موجود!")
else:
    all_files = os.listdir(MODELS_DIR)
    print(f"📁 محتوى '{MODELS_DIR}':")
    for f in sorted(all_files):
        size = os.path.getsize(os.path.join(MODELS_DIR, f))
        print(f"   {f:<55} {size/1024:.1f} KB")
    shutil.make_archive(ZIP_NAME.replace('.zip', ''), 'zip', '.', MODELS_DIR)
    size_mb = os.path.getsize(ZIP_NAME) / (1024 * 1024)
    print(f"\n✅ {ZIP_NAME} ({size_mb:.1f} MB)")
    files.download(ZIP_NAME)


📁 محتوى 'models/usdchf_technical_v8':
   USDCHF_15min.pkl                                        131.5 KB
   USDCHF_15min_metrics.json                               3.2 KB
   USDCHF_1W.pkl                                           149.7 KB
   USDCHF_1W_metrics.json                                  2.4 KB
   USDCHF_1d.pkl                                           192.3 KB
   USDCHF_1d_metrics.json                                  2.8 KB
   USDCHF_1h.pkl                                           229.5 KB
   USDCHF_1h_metrics.json                                  3.1 KB
   USDCHF_2h.pkl                                           240.9 KB
   USDCHF_2h_metrics.json                                  3.1 KB
   USDCHF_45min.pkl                                        128.3 KB
   USDCHF_45min_metrics.json                               2.8 KB
   USDCHF_4h.pkl                                           224.0 KB
   USDCHF_4h_metrics.json                                  3.1 KB
   USDCHF_5min.pkl      

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>